# Processing log files

This is for processing the data from the ``PackedRingBuffer``


In [1]:
# Helper methods and imports. Run this first and then any other cell will run
import numpy as np
import math
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import re
from datetime import datetime
import mpl_axes_aligner
#import importlib
from ReportingPlots import *  # or import Reporting.ReportingPlots as ReportingPlots

data_dir = Path("data") 

In [2]:
import importlib
import ReportingPlots  # or import Reporting.ReportingPlots as ReportingPlots
importlib.reload(ReportingPlots)

<module 'ReportingPlots' from '/home/paul/MicroMouseTools/Reporting/ReportingPlots.py'>

# Telemtry Data Load


### Copy this into the below cell to speed up selecting what to paste

Once pasted execute all below cells to get the CSV created.

``` text
pasted_log = """
<PASTE THE FULL TXT LOG CONTENTS HERE>
"""
```

In [3]:
pasted_log = """
Start Buffer
FL, LA, RA, FR, edgeLeftFast, edgeLeftSlow, edgeRightFast, edgeRightSlow, dCount, mazeLoc, fwdSpeed, rotSpeed, lVolt, rVolt, cte, steerAdj, loopTick
Robot Data: [37, 96, 10, 31, 118, 148, 11, 27, 20, 2, 2000, 0, 45, 53, 0, 0, 1606]
Robot Data: [38, 98, 10, 35, 114, 145, 11, 26, 20, 3, 2000, 0, 50, 48, 0, 0, 1607]
Robot Data: [38, 101, 12, 35, 111, 142, 11, 25, 20, 3, 2000, 0, 53, 45, 0, 0, 1608]
Robot Data: [42, 104, 13, 38, 110, 140, 11, 24, 20, 3, 2000, 0, 52, 46, 0, 0, 1609]
Robot Data: [42, 109, 14, 40, 110, 138, 12, 23, 20, 3, 2000, 0, 54, 44, 0, 0, 1610]
Robot Data: [43, 108, 13, 42, 110, 136, 12, 22, 20, 3, 2000, 0, 54, 44, 0, 0, 1611]
Robot Data: [45, 109, 15, 43, 110, 134, 13, 22, 20, 3, 2000, 0, 52, 46, 0, 0, 1612]
Robot Data: [50, 109, 16, 47, 110, 132, 14, 22, 20, 3, 2000, 0, 50, 48, 0, 0, 1613]
Robot Data: [52, 107, 17, 49, 109, 130, 15, 22, 19, 3, 2000, 0, 53, 53, 0, 0, 1614]
Robot Data: [53, 105, 19, 53, 108, 128, 16, 22, 21, 3, 2000, 0, 42, 49, 0, 0, 1615]
Robot Data: [56, 104, 22, 55, 107, 127, 17, 22, 20, 3, 2000, -303, 51, 47, 0, 0, 1616]
Robot Data: [59, 106, 21, 58, 107, 126, 18, 22, 20, 3, 2000, -606, 60, 37, 0, 0, 1617]
Robot Data: [60, 107, 25, 60, 107, 125, 19, 22, 20, 3, 2000, -909, 72, 25, 0, 0, 1618]
Robot Data: [65, 103, 26, 63, 106, 124, 20, 22, 19, 3, 2000, -1212, 88, 17, 0, 0, 1619]
Robot Data: [67, 101, 26, 65, 105, 123, 21, 22, 20, 3, 2000, -1515, 93, 6, 0, 0, 1620]
Robot Data: [71, 93, 25, 65, 103, 121, 22, 22, 21, 3, 2000, -1818, 95, 4, 0, 0, 1621]
Robot Data: [80, 80, 23, 62, 98, 118, 22, 22, 19, 3, 2000, -2121, 105, 0, 0, 0, 1622]
Robot Data: [97, 68, 17, 59, 92, 115, 21, 22, 20, 3, 2000, -2424, 99, 0, 0, 0, 1623]
Robot Data: [117, 79, 13, 55, 89, 113, 19, 21, 20, 3, 2000, -2727, 97, 2, 0, 0, 1624]
Robot Data: [127, 82, 6, 47, 88, 111, 16, 20, 19, 3, 2000, -3030, 100, 7, 0, 0, 1625]
Robot Data: [125, 86, 4, 41, 88, 109, 14, 19, 20, 3, 2000, -3108, 89, 11, 0, 0, 1626]
Robot Data: [122, 104, 0, 38, 91, 109, 11, 18, 20, 3, 2000, -3108, 88, 12, 0, 0, 1627]
Robot Data: [121, 125, 1, 34, 98, 110, 9, 17, 20, 3, 2000, -3108, 81, 19, 0, 0, 1628]
Robot Data: [121, 148, 3, 28, 108, 112, 8, 16, 20, 3, 2000, -3108, 80, 20, 0, 0, 1629]
Robot Data: [121, 170, 5, 22, 120, 116, 7, 15, 21, 3, 2000, -3108, 78, 14, 0, 0, 1630]
Robot Data: [118, 202, 5, 17, 136, 121, 7, 14, 21, 3, 2000, -3108, 79, 12, 0, 0, 1631]
Robot Data: [110, 244, 6, 13, 158, 129, 7, 14, 20, 3, 2000, -3108, 82, 15, 0, 0, 1632]
Robot Data: [99, 282, 5, 8, 183, 139, 7, 13, 20, 3, 2000, -3108, 82, 15, 0, 0, 1633]
Robot Data: [92, 268, 4, 6, 200, 147, 6, 12, 20, 3, 2000, -3108, 82, 15, 0, 0, 1634]
Robot Data: [76, 232, 2, 2, 206, 152, 5, 11, 20, 3, 2000, -3108, 86, 11, 0, 0, 1635]
Robot Data: [66, 212, 2, 1, 207, 156, 4, 10, 20, 3, 2000, -3108, 88, 10, 0, 0, 1636]
Robot Data: [53, 182, 6, 1, 202, 158, 4, 10, 20, 3, 2000, -3108, 87, 10, 0, 0, 1637]
Robot Data: [37, 143, 16, 3, 190, 157, 6, 10, 21, 3, 2000, -2805, 75, 15, 0, 0, 1638]
Robot Data: [24, 145, 33, 3, 181, 156, 11, 11, 20, 3, 2000, -2502, 70, 26, 0, 0, 1639]
Robot Data: [13, 130, 59, 4, 171, 154, 21, 14, 20, 3, 2000, -2199, 56, 41, 0, 0, 1640]
Robot Data: [7, 110, 92, 4, 159, 151, 35, 19, 21, 3, 2000, -1896, 45, 44, 0, 0, 1641]
Robot Data: [3, 94, 123, 4, 146, 147, 53, 26, 20, 3, 2000, -1593, 40, 56, 0, 0, 1642]
Robot Data: [2, 83, 151, 4, 133, 143, 73, 34, 19, 3, 2000, -1290, 35, 68, 0, 0, 1643]
Robot Data: [1, 73, 174, 4, 121, 139, 93, 43, 19, 3, 2000, -987, 37, 67, 0, 0, 1644]
Robot Data: [1, 71, 187, 4, 111, 135, 112, 52, 21, 3, 2000, -684, 30, 60, 0, 0, 1645]
Robot Data: [4, 67, 193, 3, 102, 131, 128, 61, 20, 3, 2000, -381, 35, 62, 0, 0, 1646]
Robot Data: [2, 66, 192, 4, 95, 127, 141, 69, 20, 3, 2000, -78, 35, 62, 0, 0, 1647]
Robot Data: [3, 71, 188, 5, 90, 124, 150, 76, 20, 3, 2000, 0, 41, 56, 0, 0, 1648]
Robot Data: [2, 73, 179, 5, 87, 121, 156, 82, 20, 19, 2280, 0, 53, 66, 106, 3, 1649]
Robot Data: [2, 78, 171, 5, 85, 118, 159, 88, 20, 19, 2560, 0, 72, 81, 93, 2, 1650]
Robot Data: [3, 80, 158, 5, 84, 116, 159, 92, 23, 19, 2840, 0, 80, 86, 78, 2, 1651]
Robot Data: [1, 84, 149, 6, 84, 114, 157, 96, 23, 19, 3120, 0, 101, 100, 65, 1, 1652]
Robot Data: [2, 87, 140, 6, 85, 112, 154, 99, 27, 19, 3400, 0, 111, 99, 53, 1, 1653]
Robot Data: [1, 91, 133, 6, 86, 111, 150, 101, 30, 19, 3680, 0, 116, 102, 42, 1, 1654]
Robot Data: [2, 94, 127, 5, 88, 110, 145, 103, 32, 19, 3960, 0, 128, 111, 33, 0, 1655]
Robot Data: [3, 97, 126, 6, 90, 109, 141, 104, 35, 19, 4240, 0, 133, 121, 29, 0, 1656]
Robot Data: [1, 96, 121, 5, 91, 108, 137, 105, 38, 19, 4520, 0, 137, 132, 25, 0, 1657]
Robot Data: [1, 97, 120, 4, 92, 107, 134, 106, 43, 19, 4800, 0, 134, 135, 23, 0, 1658]
Robot Data: [1, 97, 119, 4, 93, 106, 131, 107, 43, 19, 5080, 0, 147, 151, 22, 0, 1659]
Robot Data: [1, 96, 117, 2, 94, 105, 128, 108, 49, 19, 5360, 0, 141, 148, 21, 0, 1660]
Robot Data: [1, 98, 122, 2, 95, 105, 127, 109, 51, 19, 5640, 0, 157, 152, 24, 0, 1661]
Robot Data: [1, 97, 120, 3, 95, 105, 126, 110, 54, 19, 5920, 0, 164, 157, 23, 0, 1662]
Robot Data: [1, 96, 115, 3, 95, 104, 124, 110, 54, 19, 5640, 0, 149, 147, 19, 0, 1663]
Robot Data: [2, 87, 92, 4, 93, 103, 118, 109, 57, 19, 5360, 0, 120, 124, 5, 0, 1664]
Robot Data: [2, 58, 21, 5, 86, 100, 99, 104, 56, 19, 5080, 0, 99, 118, 84, 2, 1665]
Robot Data: [4, 10, 3, 7, 71, 94, 80, 98, 53, 19, 4800, 0, 97, 114, 0, -1, 1666]
Robot Data: [8, 4, 2, 12, 58, 88, 64, 92, 51, 19, 4520, 0, 93, 98, 0, 0, 1667]
Robot Data: [11, 4, 3, 18, 47, 83, 52, 86, 50, 19, 4240, 0, 79, 82, 0, 0, 1668]
Robot Data: [15, 5, 5, 22, 39, 78, 43, 81, 47, 19, 3960, 0, 75, 70, 0, 0, 1669]
Robot Data: [16, 6, 2, 26, 32, 74, 35, 76, 44, 19, 3680, 0, 68, 61, 0, 0, 1670]
Robot Data: [16, 5, 4, 27, 27, 70, 29, 72, 43, 19, 3400, 0, 54, 51, 0, 0, 1671]
Robot Data: [19, 4, 4, 32, 22, 66, 24, 68, 40, 19, 3120, 0, 45, 42, 0, 0, 1672]
Robot Data: [23, 5, 4, 35, 19, 62, 20, 64, 38, 19, 2840, 0, 33, 29, 0, 0, 1673]
Robot Data: [28, 5, 3, 41, 16, 58, 17, 60, 34, 19, 2560, 0, 27, 24, 0, 0, 1674]
Robot Data: [32, 5, 3, 46, 14, 55, 14, 56, 28, 35, 2280, 0, 30, 27, 0, 0, 1675]
Robot Data: [36, 5, 4, 49, 12, 52, 12, 53, 26, 35, 2000, 0, 23, 20, 0, 0, 1676]
Robot Data: [38, 5, 4, 52, 11, 49, 10, 50, 23, 35, 2000, 0, 30, 30, 0, 0, 1677]
Robot Data: [43, 5, 4, 56, 10, 46, 9, 47, 21, 35, 2000, 0, 34, 38, 0, 0, 1678]
Robot Data: [43, 5, 5, 59, 9, 43, 8, 44, 22, 35, 2000, 0, 30, 33, 0, 0, 1679]
Robot Data: [52, 5, 6, 65, 8, 41, 8, 42, 21, 35, 2000, 303, 31, 38, 0, 0, 1680]
Robot Data: [55, 5, 3, 68, 7, 39, 7, 40, 20, 35, 2000, 606, 23, 53, 0, 0, 1681]
Robot Data: [55, 7, 6, 72, 7, 37, 7, 38, 19, 35, 2000, 909, 15, 69, 0, 0, 1682]
Robot Data: [58, 5, 7, 77, 7, 35, 7, 36, 19, 35, 2000, 1212, 11, 73, 0, 0, 1683]
Robot Data: [57, 4, 8, 84, 6, 33, 7, 34, 18, 35, 2000, 1515, 9, 85, 0, 0, 1684]
Robot Data: [56, 6, 29, 89, 6, 31, 11, 34, 17, 35, 2000, 1818, 9, 94, 0, 0, 1685]
Robot Data: [46, 7, 60, 92, 6, 30, 21, 36, 17, 35, 2000, 2121, 8, 99, 0, 0, 1686]
Robot Data: [35, 7, 83, 92, 6, 29, 33, 39, 19, 35, 2000, 2424, 2, 96, 0, 0, 1687]
Robot Data: [6, 8, 108, 91, 6, 28, 48, 43, 18, 35, 2000, 2727, 6, 98, 0, 0, 1688]
Robot Data: [3, 10, 136, 91, 7, 27, 66, 49, 19, 35, 2000, 3030, 0, 98, 0, 0, 1689]
Robot Data: [2, 12, 166, 92, 8, 26, 86, 56, 19, 35, 2000, 3108, 3, 96, 0, 0, 1690]
Robot Data: [2, 16, 198, 91, 10, 25, 108, 65, 19, 35, 2000, 3108, 11, 89, 0, 0, 1691]
Robot Data: [4, 21, 232, 89, 12, 25, 133, 75, 19, 35, 2000, 3108, 16, 85, 0, 0, 1692]
Robot Data: [7, 29, 266, 84, 15, 25, 160, 87, 19, 35, 2000, 3108, 24, 79, 0, 0, 1693]
Robot Data: [8, 31, 320, 77, 18, 25, 192, 102, 20, 35, 2000, 3108, 23, 72, 0, 0, 1694]
Robot Data: [10, 26, 355, 67, 20, 25, 225, 118, 20, 35, 2000, 3108, 21, 75, 0, 0, 1695]
Robot Data: [12, 22, 342, 53, 20, 25, 248, 132, 19, 35, 2000, 3108, 25, 79, 0, 0, 1696]
Robot Data: [16, 20, 301, 19, 20, 25, 259, 143, 20, 35, 2000, 3108, 18, 79, 0, 0, 1697]
Robot Data: [20, 19, 269, 7, 20, 25, 261, 151, 20, 35, 2000, 3108, 16, 81, 0, 0, 1698]
Robot Data: [22, 14, 241, 7, 19, 24, 257, 157, 20, 35, 2000, 3108, 13, 84, 0, 0, 1699]
Robot Data: [25, 14, 190, 8, 18, 23, 244, 159, 20, 35, 2000, 3108, 11, 86, 0, 0, 1700]
Robot Data: [27, 13, 27, 9, 17, 22, 201, 151, 19, 35, 2000, 3108, 13, 91, 0, 0, 1701]
Robot Data: [34, 13, 2, 11, 16, 21, 161, 142, 20, 35, 2000, 2805, 17, 81, 0, 0, 1702]
Robot Data: [41, 10, 2, 14, 15, 20, 129, 133, 21, 35, 2000, 2502, 21, 69, 0, 0, 1703]
Robot Data: [40, 8, 4, 18, 14, 19, 104, 125, 20, 35, 2000, 2199, 37, 60, 0, 0, 1704]
Robot Data: [34, 7, 5, 24, 13, 18, 84, 118, 20, 35, 2000, 1896, 47, 50, 0, 0, 1705]
Robot Data: [25, 5, 8, 32, 11, 17, 69, 111, 20, 35, 2000, 1593, 57, 40, 0, 0, 1706]
Robot Data: [22, 5, 10, 36, 10, 16, 57, 105, 19, 35, 2000, 1290, 71, 34, 0, 0, 1707]
Robot Data: [22, 5, 13, 40, 9, 15, 48, 99, 21, 35, 2000, 987, 68, 23, 0, 0, 1708]
Robot Data: [24, 5, 16, 42, 8, 14, 42, 94, 20, 35, 2000, 684, 70, 27, 0, 0, 1709]
Robot Data: [25, 3, 15, 44, 7, 13, 37, 89, 20, 35, 2000, 381, 68, 29, 0, 0, 1710]
Robot Data: [27, 5, 17, 47, 7, 13, 33, 85, 20, 35, 2000, 78, 64, 32, 0, 0, 1711]
Robot Data: [27, 4, 17, 49, 6, 12, 30, 81, 19, 35, 2000, 0, 59, 45, 0, 0, 1712]
Robot Data: [31, 6, 18, 51, 6, 12, 28, 77, 20, 36, 2000, 0, 48, 50, 0, 0, 1713]
Robot Data: [30, 7, 17, 51, 6, 12, 26, 73, 20, 36, 2000, 0, 43, 55, 0, 0, 1714]
Robot Data: [34, 6, 15, 53, 6, 12, 24, 69, 20, 36, 2000, 0, 40, 58, 0, 0, 1715]
Robot Data: [35, 7, 17, 54, 6, 12, 23, 66, 19, 36, 2000, 0, 43, 63, 0, 0, 1716]
Robot Data: [41, 8, 17, 58, 6, 12, 22, 63, 20, 36, 2000, 0, 41, 58, 0, 0, 1717]
Robot Data: [43, 9, 21, 61, 7, 12, 22, 60, 20, 36, 2000, 0, 43, 56, 0, 0, 1718]
Robot Data: [46, 9, 25, 67, 7, 12, 23, 58, 20, 36, 2000, 303, 43, 56, 0, 0, 1719]
Robot Data: [46, 10, 27, 71, 8, 12, 24, 56, 20, 36, 2000, 606, 35, 64, 0, 0, 1720]
Robot Data: [48, 10, 29, 76, 8, 12, 25, 54, 21, 36, 2000, 909, 23, 69, 0, 0, 1721]
Robot Data: [47, 9, 33, 81, 8, 12, 27, 53, 19, 36, 2000, 1212, 25, 81, 0, 0, 1722]
Robot Data: [47, 12, 42, 87, 9, 12, 30, 52, 19, 36, 2000, 1515, 19, 88, 0, 0, 1723]
Robot Data: [51, 9, 51, 91, 9, 12, 34, 52, 20, 36, 2000, 1818, 11, 89, 0, 0, 1724]
Robot Data: [55, 8, 66, 92, 9, 12, 40, 53, 20, 36, 2000, 2121, 4, 96, 0, 0, 1725]
Robot Data: [52, 5, 84, 91, 8, 12, 49, 55, 20, 36, 2000, 2424, 1, 99, 0, 0, 1726]
Robot Data: [48, 4, 106, 89, 7, 12, 60, 58, 20, 36, 2000, 2727, 0, 100, 0, 0, 1727]
Robot Data: [46, 3, 130, 91, 6, 11, 74, 63, 20, 36, 2000, 3030, 0, 100, 0, 0, 1728]
Robot Data: [40, 3, 158, 92, 5, 11, 91, 69, 20, 36, 2000, 3108, 2, 98, 0, 0, 1729]
Robot Data: [29, 3, 192, 91, 5, 11, 111, 77, 20, 36, 2000, 3108, 7, 93, 0, 0, 1730]
Robot Data: [21, 3, 223, 90, 5, 11, 133, 86, 20, 36, 2000, 3108, 16, 84, 0, 0, 1731]
Robot Data: [15, 5, 252, 85, 5, 11, 157, 96, 20, 36, 2000, 3108, 23, 77, 0, 0, 1732]
Robot Data: [8, 6, 309, 76, 5, 11, 187, 109, 20, 36, 2000, 3108, 25, 75, 0, 0, 1733]
Robot Data: [5, 6, 368, 61, 5, 11, 223, 125, 21, 36, 2000, 3108, 19, 73, 0, 0, 1734]
Robot Data: [4, 4, 351, 55, 5, 11, 249, 139, 19, 36, 2000, 3108, 26, 80, 0, 0, 1735]
Robot Data: [3, 3, 309, 47, 5, 11, 261, 150, 20, 36, 2000, 3108, 19, 81, 0, 0, 1736]
Robot Data: [4, 3, 236, 39, 5, 11, 256, 155, 21, 36, 2000, 3108, 10, 82, 0, 0, 1737]
Robot Data: [3, 6, 204, 30, 5, 11, 246, 158, 20, 36, 2000, 3108, 12, 87, 0, 0, 1738]
Robot Data: [5, 10, 219, 21, 6, 11, 241, 162, 19, 36, 2000, 3108, 13, 94, 0, 0, 1739]
Robot Data: [6, 22, 202, 13, 9, 12, 233, 165, 20, 36, 2000, 3108, 10, 90, 0, 0, 1740]
Robot Data: [9, 39, 175, 6, 15, 14, 221, 166, 21, 36, 2000, 2805, 12, 80, 0, 0, 1741]
Robot Data: [6, 66, 151, 5, 25, 17, 207, 165, 20, 36, 2000, 2502, 26, 73, 0, 0, 1742]
Robot Data: [7, 97, 121, 4, 39, 22, 190, 162, 20, 36, 2000, 2199, 42, 57, 0, 0, 1743]
Robot Data: [6, 131, 93, 4, 57, 29, 171, 158, 21, 36, 2000, 1896, 45, 46, 0, 0, 1744]
Robot Data: [4, 171, 74, 6, 80, 38, 152, 153, 19, 36, 2000, 1593, 63, 43, 0, 0, 1745]
Robot Data: [7, 204, 59, 7, 105, 48, 133, 147, 20, 36, 2000, 1290, 70, 29, 0, 0, 1746]
Robot Data: [13, 229, 50, 9, 130, 59, 116, 141, 21, 36, 2000, 987, 68, 23, 0, 0, 1747]
Robot Data: [16, 248, 46, 8, 154, 71, 102, 135, 20, 36, 2000, 684, 68, 29, 0, 0, 1748]
Robot Data: [19, 252, 43, 9, 174, 82, 90, 129, 20, 36, 2000, 381, 67, 31, 0, 0, 1749]
Robot Data: [18, 251, 42, 9, 189, 93, 80, 124, 20, 36, 2000, 78, 61, 37, 0, 0, 1750]
Robot Data: [16, 249, 47, 9, 201, 103, 73, 119, 20, 36, 2000, 0, 54, 43, 0, 0, 1751]
Robot Data: [17, 244, 47, 12, 210, 112, 68, 115, 21, 20, 2280, 0, 75, 38, -256, -10, 1752]
Robot Data: [12, 236, 52, 10, 215, 120, 65, 111, 21, 20, 2560, 0, 89, 56, -184, -6, 1753]
Robot Data: [10, 224, 56, 9, 217, 127, 63, 108, 22, 20, 2840, 0, 99, 73, -168, -5, 1754]
Robot Data: [7, 208, 60, 8, 215, 132, 62, 105, 24, 20, 3120, 0, 106, 87, -148, -5, 1755]
Robot Data: [5, 194, 69, 9, 211, 136, 63, 103, 26, 20, 3400, 0, 111, 106, -125, -4, 1756]
Robot Data: [7, 180, 74, 9, 205, 139, 65, 101, 30, 20, 3680, 0, 104, 113, -106, -4, 1757]
Robot Data: [6, 169, 78, 8, 198, 141, 68, 100, 33, 20, 3960, 0, 106, 125, -91, -3, 1758]
Robot Data: [7, 157, 79, 8, 190, 142, 70, 99, 36, 20, 4240, 0, 116, 129, -78, -3, 1759]
Robot Data: [7, 152, 81, 8, 182, 143, 72, 98, 38, 20, 4520, 0, 131, 136, -71, -3, 1760]
Robot Data: [10, 149, 82, 9, 175, 143, 74, 97, 42, 20, 4800, 0, 138, 137, -67, -2, 1761]
Robot Data: [8, 150, 80, 11, 170, 143, 75, 96, 44, 20, 5080, 0, 152, 136, -70, -3, 1762]
Robot Data: [10, 143, 74, 10, 165, 143, 75, 95, 49, 20, 5360, 0, 152, 135, -69, -3, 1763]
Robot Data: [10, 141, 54, 9, 160, 143, 71, 92, 50, 20, 5640, 0, 163, 151, -87, -3, 1764]
Robot Data: [13, 130, 25, 10, 154, 142, 62, 88, 55, 20, 5920, 0, 165, 148, -60, -2, 1765]
Robot Data: [15, 144, 11, 11, 152, 142, 52, 83, 55, 20, 5640, 0, 148, 139, -88, -3, 1766]
Robot Data: [19, 137, 9, 13, 149, 142, 43, 78, 56, 20, 5360, 0, 128, 121, -74, -3, 1767]
Robot Data: [23, 128, 7, 13, 145, 141, 36, 74, 54, 20, 5080, 0, 117, 114, -56, -2, 1768]
Robot Data: [28, 122, 8, 15, 140, 140, 30, 70, 53, 20, 4800, 0, 99, 112, -1, 0, 1769]
Robot Data: [34, 114, 8, 18, 135, 138, 26, 66, 52, 20, 4520, 0, 84, 98, -1, -1, 1770]
Robot Data: [42, 108, 10, 21, 130, 136, 23, 63, 50, 20, 4240, 0, 75, 85, -1, -1, 1771]
Robot Data: [46, 103, 10, 22, 125, 134, 20, 60, 47, 20, 3960, 0, 73, 71, -1, -1, 1772]
Robot Data: [53, 104, 9, 27, 121, 132, 18, 57, 44, 20, 3680, 0, 66, 62, -1, -1, 1773]
Robot Data: [55, 105, 8, 30, 118, 130, 16, 54, 42, 20, 3400, 0, 59, 52, -1, -1, 1774]
Robot Data: [64, 103, 8, 35, 115, 128, 14, 51, 39, 20, 3120, 0, 51, 44, -1, -1, 1775]
Robot Data: [70, 103, 10, 40, 113, 126, 13, 48, 38, 20, 2840, 0, 33, 30, -1, -1, 1776]
Robot Data: [75, 103, 11, 46, 111, 125, 13, 46, 32, 20, 2560, 0, 36, 32, -1, -1, 1777]
Robot Data: [83, 106, 14, 52, 110, 124, 13, 44, 31, 4, 2600, 0, 36, 41, 0, 0, 1778]
Robot Data: [90, 105, 17, 58, 109, 123, 14, 42, 27, 4, 2600, -303, 51, 51, 0, 0, 1779]
Robot Data: [95, 108, 20, 63, 109, 122, 15, 41, 28, 4, 2600, -606, 61, 33, 0, 0, 1780]
Robot Data: [104, 106, 23, 68, 108, 121, 17, 40, 28, 4, 2600, -909, 73, 18, 0, 0, 1781]
Robot Data: [110, 103, 25, 74, 107, 120, 19, 39, 27, 4, 2600, -1212, 85, 11, 0, 0, 1782]
Robot Data: [124, 98, 27, 77, 105, 119, 21, 38, 26, 4, 2600, -1515, 93, 11, 0, 0, 1783]
Robot Data: [140, 95, 25, 77, 103, 118, 22, 37, 24, 4, 2600, -1818, 105, 14, 0, 0, 1784]
Robot Data: [162, 117, 21, 78, 106, 118, 22, 36, 25, 4, 2600, -2121, 104, 10, 0, 0, 1785]
Robot Data: [178, 121, 15, 76, 109, 118, 21, 35, 24, 4, 2600, -2424, 108, 15, 0, 0, 1786]
Robot Data: [184, 131, 12, 74, 113, 119, 19, 34, 24, 4, 2600, -2727, 107, 18, 0, 0, 1787]
Robot Data: [186, 163, 12, 67, 123, 122, 18, 33, 24, 4, 2600, -3030, 107, 20, 0, 0, 1788]
Robot Data: [187, 202, 12, 54, 139, 127, 17, 32, 25, 4, 2600, -3108, 106, 15, 0, 0, 1789]
Robot Data: [198, 245, 21, 51, 160, 134, 18, 31, 25, 4, 2600, -3108, 101, 22, 0, 0, 1790]
Robot Data: [203, 292, 22, 43, 186, 144, 19, 30, 25, 4, 2600, -3108, 96, 27, 0, 0, 1791]
Robot Data: [210, 339, 27, 37, 217, 156, 21, 30, 25, 4, 2600, -3108, 94, 31, 0, 0, 1792]
Robot Data: [212, 401, 31, 27, 254, 171, 23, 30, 26, 4, 2600, -3108, 89, 29, 0, 0, 1793]
Robot Data: [212, 478, 30, 20, 299, 190, 24, 30, 26, 4, 2600, -3108, 89, 29, 0, 0, 1794]
Robot Data: [206, 509, 27, 15, 341, 210, 25, 30, 26, 4, 2600, -3108, 91, 27, 0, 0, 1795]
Robot Data: [199, 500, 27, 11, 373, 228, 25, 30, 26, 4, 2600, -3108, 93, 25, 0, 0, 1796]
Robot Data: [186, 458, 26, 13, 390, 242, 25, 30, 25, 4, 2600, -3108, 98, 27, 0, 0, 1797]
Robot Data: [171, 423, 23, 16, 397, 253, 25, 30, 25, 4, 2600, -3108, 101, 26, 0, 0, 1798]
Robot Data: [151, 403, 24, 18, 398, 262, 25, 30, 26, 4, 2600, -3108, 99, 21, 0, 0, 1799]
Robot Data: [126, 377, 19, 18, 394, 269, 24, 29, 26, 4, 2600, -3108, 101, 20, 0, 0, 1800]
Robot Data: [89, 305, 20, 21, 376, 271, 23, 28, 26, 4, 2600, -2805, 92, 28, 0, 0, 1801]
Robot Data: [64, 272, 15, 25, 355, 271, 21, 27, 26, 4, 2600, -2502, 84, 36, 0, 0, 1802]
Robot Data: [37, 261, 16, 30, 336, 270, 20, 26, 26, 4, 2600, -2199, 69, 51, 0, 0, 1803]
Robot Data: [20, 224, 13, 33, 314, 267, 19, 25, 26, 4, 2600, -1896, 60, 60, 0, 0, 1804]
Robot Data: [19, 184, 9, 31, 288, 262, 17, 24, 26, 4, 2600, -1593, 52, 68, 0, 0, 1805]
Robot Data: [36, 157, 9, 28, 262, 255, 15, 23, 25, 4, 2600, -1290, 51, 77, 0, 0, 1806]
Robot Data: [46, 140, 7, 27, 238, 248, 13, 22, 26, 4, 2600, -987, 43, 78, 0, 0, 1807]
Robot Data: [50, 127, 7, 27, 216, 240, 12, 21, 26, 4, 2600, -684, 44, 77, 0, 0, 1808]
Robot Data: [54, 122, 9, 30, 197, 233, 11, 20, 26, 4, 2320, -381, 29, 62, 0, 0, 1809]
Robot Data: [54, 120, 8, 32, 182, 226, 10, 19, 24, 4, 2040, -78, 22, 51, 0, 0, 1810]
Robot Data: [55, 127, 10, 33, 171, 220, 10, 18, 23, 4, 2000, 0, 28, 46, 0, 0, 1811]
Robot Data: [61, 133, 12, 39, 163, 215, 10, 18, 21, 5, 2000, 0, 43, 44, 0, 0, 1812]
Robot Data: [62, 139, 13, 41, 158, 210, 11, 18, 21, 5, 2000, 0, 49, 37, 0, 0, 1813]
Robot Data: [64, 147, 17, 47, 156, 206, 12, 18, 21, 5, 2000, -303, 59, 26, 0, 0, 1814]
Robot Data: [68, 153, 17, 49, 155, 203, 13, 18, 21, 5, 2000, -606, 69, 15, 0, 0, 1815]
Robot Data: [69, 153, 20, 51, 155, 200, 14, 18, 19, 5, 2000, -909, 86, 12, 0, 0, 1816]
Robot Data: [72, 141, 18, 51, 152, 196, 15, 18, 19, 5, 2000, -1212, 93, 6, 0, 0, 1817]
Robot Data: [77, 127, 18, 51, 147, 192, 16, 18, 20, 5, 2000, -1515, 90, 2, 0, 0, 1818]
Robot Data: [81, 105, 16, 49, 139, 187, 16, 18, 19, 5, 2000, -1818, 93, 7, 0, 0, 1819]
Robot Data: [92, 82, 11, 46, 128, 180, 15, 18, 18, 5, 2000, -2121, 94, 15, 0, 0, 1820]
Robot Data: [106, 64, 8, 45, 115, 173, 14, 17, 19, 5, 2000, -2424, 90, 14, 0, 0, 1821]
Robot Data: [112, 73, 6, 40, 107, 167, 12, 16, 20, 5, 2000, -2727, 86, 10, 0, 0, 1822]
Robot Data: [109, 90, 3, 36, 104, 162, 10, 15, 19, 5, 2000, -3030, 95, 10, 0, 0, 1823]
Robot Data: [103, 90, 1, 33, 101, 158, 8, 14, 20, 5, 2000, -3108, 95, 3, 0, 0, 1824]
Robot Data: [103, 101, 0, 30, 101, 154, 6, 13, 20, 5, 2000, -3108, 92, 6, 0, 0, 1825]
Robot Data: [103, 118, 1, 26, 104, 152, 5, 12, 19, 5, 2000, -3108, 92, 13, 0, 0, 1826]
Robot Data: [100, 137, 1, 21, 111, 151, 4, 11, 20, 5, 2000, -3108, 84, 15, 0, 0, 1827]
Robot Data: [95, 164, 0, 15, 122, 152, 3, 10, 21, 5, 2000, -3108, 79, 12, 0, 0, 1828]
Robot Data: [86, 202, 0, 9, 138, 155, 2, 9, 20, 5, 2000, -3108, 79, 19, 0, 0, 1829]
Robot Data: [77, 234, 2, 6, 157, 160, 2, 9, 20, 5, 2000, -3108, 79, 19, 0, 0, 1830]
Robot Data: [70, 206, 1, 5, 167, 163, 2, 9, 20, 5, 2000, -3108, 83, 15, 0, 0, 1831]
Robot Data: [58, 176, 3, 4, 169, 164, 2, 9, 20, 5, 2000, -3108, 81, 17, 0, 0, 1832]
Robot Data: [49, 159, 7, 2, 167, 164, 3, 9, 21, 5, 2000, -3108, 79, 11, 0, 0, 1833]
Robot Data: [37, 145, 15, 1, 163, 163, 5, 9, 19, 5, 2000, -3108, 90, 15, 0, 0, 1834]
Robot Data: [26, 129, 33, 0, 156, 161, 11, 11, 20, 5, 2000, -3108, 84, 14, 0, 0, 1835]
Robot Data: [19, 110, 59, 1, 147, 158, 21, 14, 20, 5, 2000, -2805, 78, 20, 0, 0, 1836]
Robot Data: [7, 100, 94, 1, 138, 154, 36, 19, 20, 5, 2000, -2502, 71, 27, 0, 0, 1837]
Robot Data: [3, 89, 142, 0, 128, 150, 57, 27, 21, 5, 2000, -2199, 54, 36, 0, 0, 1838]
Robot Data: [1, 75, 195, 0, 117, 145, 85, 38, 20, 5, 2000, -1896, 50, 46, 0, 0, 1839]
Robot Data: [1, 62, 246, 1, 106, 140, 117, 51, 20, 5, 2000, -1593, 48, 49, 0, 0, 1840]
Robot Data: [0, 52, 292, 2, 95, 135, 152, 66, 20, 5, 2000, -1290, 37, 60, 0, 0, 1841]
Robot Data: [1, 43, 328, 3, 85, 129, 187, 82, 20, 5, 2000, -987, 33, 64, 0, 0, 1842]
Robot Data: [0, 40, 352, 4, 76, 123, 220, 99, 20, 5, 2000, -684, 32, 65, 0, 0, 1843]
Robot Data: [1, 36, 363, 5, 68, 118, 249, 116, 20, 5, 2000, -381, 32, 65, 0, 0, 1844]
Robot Data: [0, 38, 366, 5, 62, 113, 272, 132, 20, 5, 2000, -78, 32, 65, 0, 0, 1845]
Robot Data: [0, 38, 360, 5, 57, 108, 290, 146, 20, 5, 2000, 0, 39, 57, 0, 0, 1846]
Robot Data: [2, 40, 353, 5, 54, 104, 303, 159, 20, 21, 2280, 0, 28, 91, 255, 16, 1847]
Robot Data: [0, 42, 341, 3, 52, 100, 311, 170, 21, 21, 2560, 0, 40, 105, 255, 8, 1848]
Robot Data: [1, 45, 316, 2, 51, 97, 312, 179, 22, 21, 2840, 0, 62, 110, 255, 8, 1849]
Robot Data: [0, 54, 281, 1, 52, 94, 306, 185, 24, 21, 3120, 0, 85, 108, 227, 6, 1850]
Robot Data: [1, 60, 242, 1, 54, 92, 293, 189, 27, 21, 3400, 0, 107, 102, 182, 5, 1851]
Robot Data: [1, 67, 213, 2, 57, 90, 277, 191, 29, 21, 3680, 0, 128, 96, 146, 4, 1852]
Robot Data: [1, 76, 191, 1, 61, 89, 260, 191, 32, 21, 3960, 0, 143, 96, 115, 3, 1853]
Robot Data: [3, 79, 169, 1, 65, 88, 242, 190, 36, 21, 4240, 0, 143, 103, 90, 2, 1854]
Robot Data: [2, 84, 166, 0, 69, 88, 227, 189, 38, 21, 4520, 0, 146, 122, 82, 2, 1855]
Robot Data: [1, 83, 160, 0, 72, 88, 214, 187, 42, 21, 4800, 0, 139, 136, 77, 2, 1856]
Robot Data: [2, 83, 161, 0, 74, 88, 203, 185, 45, 21, 5080, 0, 132, 149, 78, 2, 1857]
Robot Data: [1, 83, 164, 0, 76, 88, 195, 184, 48, 21, 5360, 0, 139, 156, 81, 2, 1858]
Robot Data: [3, 80, 170, 0, 77, 88, 190, 183, 53, 21, 5640, 0, 145, 146, 90, 2, 1859]
Robot Data: [2, 79, 171, 1, 77, 87, 186, 182, 51, 21, 5920, 0, 167, 174, 92, 2, 1860]
Robot Data: [1, 82, 169, 0, 78, 87, 183, 181, 56, 21, 6200, 0, 166, 176, 87, 2, 1861]
Robot Data: [3, 81, 162, 0, 79, 87, 179, 180, 58, 21, 6480, 0, 165, 191, 81, 2, 1862]
Robot Data: [1, 83, 156, 1, 80, 87, 174, 179, 61, 21, 6760, 0, 175, 195, 73, 2, 1863]
Robot Data: [2, 76, 150, 0, 79, 86, 169, 177, 67, 21, 7040, 0, 178, 182, 74, 2, 1864]
Robot Data: [3, 79, 161, 1, 79, 86, 167, 176, 66, 21, 7320, 0, 204, 199, 82, 2, 1865]
Robot Data: [2, 83, 157, 1, 80, 86, 165, 175, 69, 21, 7600, 0, 210, 207, 74, 2, 1866]
Robot Data: [4, 88, 151, 0, 82, 86, 162, 174, 73, 21, 7880, 0, 214, 203, 63, 1, 1867]
Robot Data: [1, 89, 143, 1, 83, 86, 158, 172, 74, 21, 8160, 0, 224, 222, 54, 1, 1868]
Robot Data: [3, 90, 136, 0, 84, 86, 154, 170, 80, 21, 8440, 0, 218, 219, 46, 1, 1869]
Robot Data: [2, 93, 134, 1, 86, 86, 150, 168, 82, 21, 8720, 0, 224, 232, 41, 1, 1870]
Robot Data: [1, 96, 133, 0, 88, 87, 147, 166, 83, 21, 9000, 0, 235, 250, 37, 1, 1871]
Robot Data: [3, 96, 122, 0, 90, 88, 142, 163, 88, 21, 9000, 0, 220, 234, 26, 0, 1872]
Robot Data: [2, 98, 123, 1, 92, 89, 138, 161, 90, 21, 9000, 0, 223, 217, 25, 0, 1873]
Robot Data: [3, 97, 119, 0, 93, 90, 134, 158, 92, 21, 9000, 0, 224, 201, 22, 0, 1874]
Robot Data: [3, 102, 117, 1, 95, 91, 131, 155, 88, 21, 9000, 0, 239, 215, 15, 0, 1875]
Robot Data: [3, 103, 114, 0, 97, 92, 128, 152, 92, 21, 9000, 0, 216, 209, 11, 0, 1876]
Robot Data: [1, 104, 107, 0, 98, 93, 124, 149, 91, 21, 9000, 0, 218, 212, 3, 0, 1877]
Robot Data: [2, 100, 106, 0, 98, 93, 120, 146, 91, 21, 9000, 0, 210, 219, 6, 0, 1878]
Robot Data: [4, 107, 106, 1, 100, 94, 117, 144, 91, 21, 9000, 0, 211, 218, -1, -1, 1879]
Robot Data: [2, 113, 106, 1, 103, 95, 115, 142, 93, 21, 9000, 0, 204, 207, -7, -1, 1880]
Robot Data: [2, 115, 104, 1, 105, 96, 113, 140, 93, 21, 9000, 0, 204, 204, -11, -1, 1881]
Robot Data: [2, 117, 103, 0, 107, 97, 111, 138, 93, 21, 9000, 0, 204, 201, -14, -1, 1882]
Robot Data: [4, 120, 101, 1, 110, 98, 109, 136, 93, 21, 9000, 0, 203, 199, -19, -1, 1883]
Robot Data: [2, 124, 98, 1, 113, 100, 107, 134, 92, 21, 9000, 0, 207, 199, -26, -1, 1884]
Robot Data: [3, 126, 96, 0, 116, 102, 105, 132, 92, 21, 9000, 0, 205, 199, -30, -1, 1885]
Robot Data: [5, 127, 94, 1, 118, 104, 103, 130, 94, 21, 9000, 0, 198, 188, -33, -1, 1886]
Robot Data: [4, 128, 92, 1, 120, 106, 101, 128, 94, 21, 9000, 0, 196, 186, -36, -2, 1887]
Robot Data: [4, 128, 97, 1, 122, 107, 100, 126, 92, 21, 9000, 0, 199, 193, -31, -1, 1888]
Robot Data: [2, 118, 97, 1, 121, 108, 99, 124, 92, 21, 9000, 0, 199, 192, -21, -1, 1889]
Robot Data: [3, 128, 98, 0, 122, 109, 99, 122, 92, 21, 9000, 0, 196, 193, -30, -1, 1890]
Robot Data: [4, 135, 97, 1, 125, 111, 99, 120, 92, 21, 9000, 0, 195, 191, -38, -2, 1891]
Robot Data: [1, 140, 98, 1, 128, 113, 99, 119, 91, 21, 9000, 0, 199, 192, -42, -2, 1892]
Robot Data: [4, 140, 98, 1, 130, 115, 99, 118, 91, 21, 9000, 0, 201, 190, -42, -2, 1893]
Robot Data: [4, 141, 97, 1, 132, 117, 99, 117, 91, 21, 9000, 0, 198, 192, -44, -2, 1894]
Robot Data: [4, 140, 94, 0, 134, 118, 98, 116, 92, 21, 9000, 0, 194, 187, -46, -2, 1895]
Robot Data: [4, 139, 93, 0, 135, 119, 97, 115, 91, 21, 9000, 0, 196, 190, -46, -2, 1896]
Robot Data: [2, 138, 91, 1, 136, 120, 96, 114, 90, 21, 9000, 0, 198, 195, -47, -2, 1897]
Robot Data: [4, 138, 93, 0, 136, 121, 95, 113, 91, 21, 9000, 0, 192, 193, -45, -2, 1898]
Robot Data: [4, 137, 94, 0, 136, 122, 95, 112, 91, 21, 9000, 0, 192, 192, -43, -2, 1899]
Robot Data: [3, 129, 94, 1, 135, 122, 95, 111, 90, 21, 9000, 0, 197, 193, -35, -2, 1900]
Robot Data: [3, 152, 98, 1, 138, 124, 96, 110, 91, 21, 9000, 0, 193, 190, -54, -2, 1901]
Robot Data: [3, 149, 100, 1, 140, 126, 97, 109, 90, 21, 9000, 0, 198, 191, -49, -2, 1902]
Robot Data: [3, 147, 98, 1, 141, 127, 97, 108, 90, 21, 9000, 0, 200, 189, -49, -2, 1903]
Robot Data: [4, 147, 104, 1, 142, 128, 98, 108, 91, 21, 9000, 0, 192, 190, -43, -2, 1904]
Robot Data: [2, 146, 100, 2, 143, 129, 98, 108, 90, 21, 9000, 0, 194, 194, -46, -2, 1905]
Robot Data: [1, 142, 99, 1, 143, 130, 98, 107, 90, 21, 9000, 0, 194, 194, -43, -2, 1906]
Robot Data: [3, 140, 101, 1, 142, 131, 99, 107, 91, 21, 9000, 0, 194, 186, -39, -2, 1907]
Robot Data: [4, 138, 102, 0, 141, 131, 100, 107, 90, 21, 9000, 0, 199, 188, -36, -2, 1908]
Robot Data: [4, 135, 101, 1, 140, 131, 100, 107, 89, 21, 9000, 0, 201, 194, -34, -2, 1909]
Robot Data: [1, 133, 100, 1, 139, 131, 100, 107, 91, 21, 9000, 0, 188, 193, -33, -1, 1910]
Robot Data: [2, 130, 93, 1, 137, 131, 99, 106, 90, 21, 9000, 0, 194, 193, -37, -2, 1911]
Robot Data: [2, 136, 98, 1, 137, 131, 99, 106, 90, 21, 9000, 0, 196, 192, -38, -2, 1912]
Robot Data: [3, 129, 99, 1, 135, 131, 99, 106, 90, 21, 9000, 0, 191, 196, -30, -1, 1913]
Robot Data: [2, 131, 104, 1, 134, 131, 100, 106, 90, 21, 9000, 0, 194, 193, -27, -1, 1914]
Robot Data: [0, 132, 102, 1, 134, 131, 100, 106, 90, 21, 9000, 0, 196, 192, -30, -1, 1915]
Robot Data: [1, 130, 103, 1, 133, 131, 101, 106, 90, 21, 9000, 0, 193, 194, -27, -1, 1916]
Robot Data: [2, 129, 102, 1, 132, 131, 101, 106, 90, 21, 9000, 0, 194, 194, -27, -1, 1917]
Robot Data: [1, 130, 104, 1, 132, 131, 102, 106, 90, 21, 9000, 0, 196, 192, -26, -1, 1918]
Robot Data: [3, 128, 104, 1, 131, 131, 102, 106, 90, 21, 9000, 0, 197, 190, -24, -1, 1919]
Robot Data: [2, 126, 103, 1, 130, 131, 102, 106, 90, 21, 8720, 0, 178, 179, -23, -1, 1920]
Robot Data: [4, 127, 102, 2, 129, 131, 102, 106, 89, 21, 8440, 0, 165, 165, -25, -1, 1921]
Robot Data: [2, 124, 98, 2, 128, 131, 101, 106, 86, 21, 8160, 0, 163, 155, -26, -1, 1922]
Robot Data: [4, 106, 103, 3, 124, 129, 101, 106, 84, 21, 7880, 0, 152, 145, -3, -1, 1923]
Robot Data: [2, 112, 108, 3, 122, 128, 102, 106, 82, 21, 7600, 0, 145, 138, -4, -1, 1924]
Robot Data: [4, 112, 109, 2, 120, 127, 103, 106, 80, 21, 7320, 0, 130, 131, -3, -1, 1925]
Robot Data: [4, 111, 108, 3, 118, 126, 104, 106, 77, 21, 7040, 0, 123, 123, -3, -1, 1926]
Robot Data: [6, 111, 111, 3, 117, 125, 105, 106, 74, 21, 6760, 0, 116, 116, 0, 0, 1927]
Robot Data: [4, 112, 112, 4, 116, 124, 106, 106, 72, 21, 6480, 0, 104, 104, 0, 0, 1928]
Robot Data: [7, 111, 111, 4, 115, 123, 107, 106, 68, 21, 6200, 0, 104, 104, 0, 0, 1929]
Robot Data: [6, 113, 106, 5, 115, 122, 107, 106, 65, 21, 5920, 0, 95, 99, -7, -1, 1930]
Robot Data: [7, 115, 103, 6, 115, 122, 106, 106, 60, 21, 5640, 0, 98, 97, -12, -1, 1931]
Robot Data: [10, 116, 104, 7, 115, 122, 106, 106, 59, 21, 5360, 0, 90, 78, -12, -1, 1932]
Robot Data: [10, 115, 101, 8, 115, 122, 105, 106, 57, 21, 5080, 0, 80, 66, -14, -1, 1933]
Robot Data: [11, 116, 100, 9, 115, 122, 104, 106, 53, 21, 4800, 0, 80, 67, -16, -1, 1934]
Robot Data: [14, 116, 96, 10, 115, 122, 102, 105, 50, 21, 4520, 0, 72, 62, -20, -1, 1935]
Robot Data: [14, 111, 70, 12, 114, 121, 96, 103, 46, 21, 4240, 0, 67, 61, -41, -2, 1936]
Robot Data: [19, 117, 17, 12, 115, 121, 80, 98, 43, 21, 3960, 0, 62, 55, -34, -2, 1937]
Robot Data: [22, 125, 7, 13, 117, 121, 65, 92, 42, 21, 3680, 0, 44, 45, -50, -2, 1938]
Robot Data: [22, 123, 3, 15, 118, 121, 53, 86, 37, 21, 3400, 0, 51, 47, -46, -2, 1939]
Robot Data: [26, 122, 4, 16, 119, 121, 43, 81, 36, 21, 3120, 0, 34, 39, -1, 0, 1940]
Robot Data: [32, 121, 5, 18, 119, 121, 35, 76, 32, 21, 2840, 0, 32, 35, -1, -1, 1941]
Robot Data: [35, 122, 7, 21, 120, 121, 29, 72, 30, 21, 2560, 0, 24, 24, -1, -1, 1942]
Robot Data: [35, 117, 5, 21, 119, 121, 24, 68, 25, 21, 2280, 0, 33, 18, -1, -1, 1943]
Robot Data: [38, 117, 5, 24, 119, 121, 20, 64, 23, 21, 2000, 0, 25, 15, -1, -1, 1944]
Robot Data: [37, 115, 6, 22, 118, 121, 17, 60, 21, 21, 2000, 0, 25, 27, -1, -1, 1945]
Robot Data: [41, 115, 5, 26, 117, 121, 15, 57, 20, 21, 2000, 0, 28, 31, -1, -1, 1946]
Robot Data: [42, 112, 6, 26, 116, 120, 13, 54, 19, 133, 2000, 0, 30, 37, 0, 0, 1947]
Robot Data: [44, 111, 7, 27, 115, 119, 12, 51, 18, 133, 2000, 0, 34, 41, 0, 0, 1948]
Robot Data: [43, 113, 8, 27, 115, 119, 11, 48, 18, 133, 2000, 0, 36, 42, 0, 0, 1949]
Robot Data: [48, 112, 6, 32, 114, 119, 10, 45, 17, 133, 2000, 0, 44, 43, 0, 0, 1950]
Robot Data: [47, 118, 8, 33, 115, 119, 10, 43, 18, 133, 2000, 0, 44, 40, 0, 0, 1951]
Robot Data: [47, 118, 11, 35, 116, 119, 10, 41, 18, 133, 2000, 0, 46, 39, 0, 0, 1952]
Robot Data: [51, 120, 10, 37, 117, 119, 10, 39, 18, 133, 2000, 0, 47, 41, 0, 0, 1953]
Robot Data: [54, 118, 10, 39, 117, 119, 10, 37, 17, 133, 2000, 0, 56, 42, 0, 0, 1954]
Robot Data: [54, 117, 10, 39, 117, 119, 10, 35, 17, 133, 2000, 0, 55, 46, 0, 0, 1955]
Robot Data: [55, 112, 10, 42, 116, 119, 10, 33, 18, 133, 2000, 0, 49, 47, 0, 0, 1956]
Robot Data: [58, 113, 12, 41, 115, 119, 10, 32, 18, 133, 2000, 0, 49, 50, 0, 0, 1957]
Robot Data: [62, 111, 13, 46, 114, 119, 11, 31, 19, 133, 2000, 0, 43, 51, 0, 0, 1958]
Robot Data: [67, 110, 16, 49, 113, 118, 12, 30, 19, 133, 2000, 0, 44, 51, 0, 0, 1959]
Robot Data: [67, 110, 17, 50, 112, 118, 13, 29, 19, 133, 2000, -303, 50, 46, 0, 0, 1960]
Robot Data: [69, 107, 17, 52, 111, 117, 14, 28, 19, 133, 2000, -606, 62, 35, 0, 0, 1961]
Robot Data: [76, 108, 16, 55, 110, 116, 14, 27, 20, 133, 2000, -909, 70, 20, 0, 0, 1962]
Robot Data: [76, 105, 20, 57, 109, 115, 15, 27, 19, 133, 2000, -1212, 84, 14, 0, 0, 1963]
Robot Data: [84, 98, 20, 59, 107, 114, 16, 27, 19, 133, 2000, -1515, 93, 6, 0, 0, 1964]
Robot Data: [93, 86, 17, 56, 103, 112, 16, 26, 19, 133, 2000, -1818, 97, 3, 0, 0, 1965]
Robot Data: [110, 72, 14, 55, 97, 110, 16, 25, 19, 133, 2000, -2121, 97, 4, 0, 0, 1966]
Robot Data: [125, 77, 12, 52, 93, 108, 15, 24, 20, 133, 2000, -2424, 90, 4, 0, 0, 1967]
Robot Data: [128, 92, 7, 45, 93, 107, 13, 23, 19, 133, 2000, -2727, 94, 8, 0, 0, 1968]
Robot Data: [121, 89, 4, 42, 92, 106, 11, 22, 20, 133, 2000, -3030, 91, 4, 0, 0, 1969]
Robot Data: [117, 99, 1, 39, 93, 106, 9, 21, 19, 133, 2000, -3108, 91, 12, 0, 0, 1970]
Robot Data: [114, 121, 2, 35, 99, 107, 8, 20, 20, 133, 2000, -3108, 87, 10, 0, 0, 1971]
Robot Data: [118, 141, 0, 28, 107, 109, 6, 19, 20, 133, 2000, -3108, 81, 16, 0, 0, 1972]
Robot Data: [114, 163, 0, 22, 118, 112, 5, 18, 21, 133, 2000, -3108, 76, 13, 0, 0, 1973]
Robot Data: [111, 192, 1, 15, 133, 117, 4, 17, 20, 133, 2000, -3108, 78, 18, 0, 0, 1974]
Robot Data: [105, 235, 0, 11, 153, 124, 3, 16, 20, 133, 2000, -3108, 76, 20, 0, 0, 1975]
Robot Data: [94, 272, 0, 7, 177, 133, 2, 15, 21, 133, 2000, -3108, 74, 13, 0, 0, 1976]
Robot Data: [86, 262, 3, 5, 194, 141, 2, 14, 20, 133, 2000, -3108, 83, 11, 0, 0, 1977]
Robot Data: [74, 229, 3, 3, 201, 147, 2, 13, 20, 133, 2000, -3108, 82, 12, 0, 0, 1978]
Robot Data: [65, 197, 5, 3, 200, 150, 3, 13, 20, 133, 2000, -3108, 84, 10, 0, 0, 1979]
Robot Data: [50, 174, 11, 1, 195, 152, 5, 13, 20, 133, 2000, -3108, 82, 12, 0, 0, 1980]
Robot Data: [35, 141, 27, 1, 184, 151, 9, 14, 20, 133, 2000, -3108, 84, 10, 0, 0, 1981]
Robot Data: [19, 143, 51, 0, 176, 151, 17, 16, 21, 133, 2000, -2805, 72, 15, 0, 0, 1982]
Robot Data: [10, 125, 84, 0, 166, 149, 30, 20, 20, 133, 2000, -2502, 67, 26, 0, 0, 1983]
Robot Data: [6, 105, 125, 0, 154, 146, 49, 27, 20, 133, 2000, -2199, 54, 39, 0, 0, 1984]
Robot Data: [1, 89, 173, 1, 141, 142, 74, 36, 20, 133, 2000, -1896, 47, 46, 0, 0, 1985]
Robot Data: [1, 74, 217, 1, 128, 138, 103, 47, 20, 133, 2000, -1593, 41, 53, 0, 0, 1986]
Robot Data: [1, 62, 246, 1, 115, 133, 132, 59, 20, 133, 2000, -1290, 34, 60, 0, 0, 1987]
Robot Data: [0, 56, 274, 2, 103, 128, 160, 72, 20, 133, 2000, -987, 33, 60, 0, 0, 1988]
Robot Data: [0, 50, 294, 2, 92, 123, 187, 86, 20, 133, 2000, -684, 34, 60, 0, 0, 1989]
Robot Data: [1, 47, 302, 2, 83, 118, 210, 100, 19, 133, 2000, -381, 37, 64, 0, 0, 1990]
Robot Data: [0, 48, 304, 3, 76, 114, 229, 113, 20, 133, 2000, -78, 36, 59, 0, 0, 1991]
Robot Data: [0, 50, 306, 3, 71, 110, 244, 125, 20, 133, 2000, 0, 39, 55, 0, 0, 1992]
Robot Data: [1, 50, 297, 2, 67, 106, 255, 136, 20, 132, 2280, 0, 43, 75, 247, 8, 1993]
Robot Data: [1, 53, 286, 2, 64, 103, 261, 145, 21, 132, 2560, 0, 52, 90, 233, 6, 1994]
Robot Data: [1, 56, 269, 1, 62, 100, 263, 153, 23, 132, 2840, 0, 67, 95, 213, 6, 1995]
Robot Data: [1, 62, 248, 0, 62, 98, 260, 159, 24, 132, 3120, 0, 86, 104, 186, 5, 1996]
Robot Data: [0, 69, 231, 1, 63, 96, 254, 164, 26, 132, 3400, 0, 106, 107, 162, 4, 1997]
Robot Data: [0, 77, 207, 1, 66, 95, 245, 167, 29, 132, 3680, 0, 121, 101, 130, 3, 1998]
Robot Data: [1, 80, 186, 1, 69, 94, 233, 168, 33, 132, 3960, 0, 129, 100, 106, 3, 1999]
Robot Data: [2, 83, 170, 0, 72, 93, 220, 168, 35, 132, 4240, 0, 137, 114, 87, 2, 2000]
Robot Data: [1, 88, 159, 0, 75, 93, 208, 167, 38, 132, 4520, 0, 143, 123, 71, 2, 2001]
Robot Data: [1, 89, 153, 0, 78, 93, 197, 166, 42, 132, 4800, 0, 139, 134, 64, 1, 2002]
Robot Data: [1, 89, 148, 0, 80, 93, 187, 165, 45, 132, 5080, 0, 137, 142, 59, 1, 2003]
Robot Data: [1, 90, 146, 1, 82, 93, 179, 164, 48, 132, 5360, 0, 143, 150, 56, 1, 2004]
Robot Data: [1, 92, 140, 1, 84, 93, 171, 163, 50, 132, 5640, 0, 153, 160, 48, 1, 2005]
Robot Data: [0, 92, 137, 0, 86, 93, 164, 161, 56, 132, 5920, 0, 146, 157, 45, 1, 2006]
Robot Data: [0, 89, 136, 0, 87, 93, 158, 159, 57, 132, 6200, 0, 168, 162, 47, 1, 2007]
Robot Data: [2, 89, 141, 0, 87, 93, 155, 158, 57, 132, 6480, 0, 181, 177, 52, 1, 2008]
Robot Data: [1, 84, 151, 0, 86, 92, 154, 158, 60, 132, 6760, 0, 184, 189, 67, 2, 2009]
Robot Data: [1, 87, 144, 0, 86, 92, 152, 157, 65, 132, 7040, 0, 181, 192, 57, 1, 2010]
Robot Data: [1, 90, 140, 0, 87, 92, 150, 156, 67, 132, 7320, 0, 188, 206, 50, 1, 2011]
Robot Data: [2, 95, 135, 0, 89, 92, 147, 155, 70, 132, 7600, 0, 197, 210, 40, 1, 2012]
Robot Data: [1, 98, 131, 0, 91, 92, 144, 154, 73, 132, 7880, 0, 206, 208, 33, 0, 2013]
Robot Data: [1, 98, 126, 0, 92, 92, 140, 152, 75, 132, 8160, 0, 223, 211, 28, 0, 2014]
Robot Data: [1, 103, 122, 0, 94, 93, 136, 150, 79, 132, 8440, 0, 223, 217, 19, 0, 2015]
Robot Data: [1, 104, 114, 1, 96, 94, 132, 148, 81, 132, 8720, 0, 232, 229, 10, 0, 2016]
Robot Data: [1, 107, 111, 0, 98, 95, 128, 146, 84, 132, 9000, 0, 239, 236, 4, 0, 2017]
Robot Data: [3, 109, 108, 0, 100, 96, 124, 144, 90, 132, 9000, 0, 219, 216, -1, -1, 2018]
Robot Data: [1, 111, 103, 1, 102, 97, 120, 141, 88, 132, 9000, 0, 225, 226, -8, -1, 2019]
Robot Data: [1, 113, 100, 0, 104, 98, 116, 138, 89, 132, 9000, 0, 223, 223, -13, -1, 2020]
Robot Data: [1, 118, 100, 0, 107, 99, 113, 136, 90, 132, 9000, 0, 223, 215, -18, -1, 2021]
Robot Data: [3, 122, 107, 1, 110, 100, 112, 134, 93, 132, 9000, 0, 209, 206, -15, -1, 2022]
Robot Data: [3, 120, 99, 0, 112, 101, 109, 132, 93, 132, 9000, 0, 206, 206, -21, -1, 2023]
Robot Data: [2, 143, 95, 1, 118, 104, 106, 130, 92, 132, 9000, 0, 212, 204, -48, -2, 2024]
Robot Data: [4, 149, 92, 0, 124, 107, 103, 128, 92, 132, 9000, 0, 214, 200, -57, -2, 2025]
Robot Data: [3, 152, 89, 1, 130, 110, 100, 126, 92, 132, 9000, 0, 213, 199, -63, -2, 2026]
Robot Data: [5, 159, 87, 2, 136, 113, 97, 124, 93, 132, 9000, 0, 206, 196, -72, -3, 2027]
Robot Data: [6, 167, 80, 1, 142, 116, 94, 121, 92, 132, 9000, 0, 208, 198, -87, -3, 2028]
Robot Data: [5, 173, 76, 1, 148, 120, 90, 118, 92, 132, 9000, 0, 207, 197, -97, -3, 2029]
Robot Data: [5, 176, 71, 2, 154, 124, 86, 115, 92, 132, 9000, 0, 208, 194, -105, -4, 2030]
Robot Data: [7, 186, 70, 1, 160, 128, 83, 112, 94, 132, 9000, 0, 199, 185, -116, -4, 2031]
Robot Data: [8, 193, 60, 2, 167, 132, 78, 109, 93, 132, 8720, 0, 191, 166, -133, -5, 2032]
Robot Data: [6, 196, 13, 2, 173, 136, 65, 103, 91, 132, 8440, 0, 187, 148, -192, -6, 2033]
Robot Data: [7, 200, 5, 3, 178, 140, 53, 97, 89, 132, 8160, 0, 176, 136, -200, -7, 2034]
Robot Data: [7, 181, 4, 3, 179, 143, 43, 91, 86, 132, 7880, 0, 160, 136, -162, -5, 2035]
Robot Data: [6, 184, 7, 3, 180, 146, 36, 86, 84, 132, 7600, 0, 151, 128, -168, -6, 2036]
Robot Data: [3, 181, 7, 3, 180, 148, 30, 81, 80, 132, 7320, 0, 139, 132, -162, -5, 2037]
Robot Data: [6, 179, 8, 3, 180, 150, 26, 76, 78, 132, 7040, 0, 119, 129, -158, -5, 2038]
Robot Data: [4, 180, 11, 4, 180, 152, 23, 72, 74, 132, 6760, 0, 113, 127, -160, -5, 2039]
Robot Data: [4, 180, 15, 5, 180, 154, 21, 68, 73, 132, 6480, 0, 100, 109, -160, -5, 2040]
Robot Data: [5, 184, 20, 5, 181, 156, 21, 65, 68, 132, 6200, 0, 114, 101, -168, -6, 2041]
Robot Data: [5, 185, 26, 6, 182, 158, 22, 63, 67, 132, 5920, 0, 106, 80, -170, -6, 2042]
Robot Data: [7, 185, 39, 7, 183, 160, 25, 62, 63, 132, 5640, 0, 106, 71, -170, -6, 2043]
Robot Data: [7, 186, 48, 8, 184, 162, 30, 61, 59, 132, 5360, 0, 100, 70, -172, -6, 2044]
Robot Data: [7, 184, 47, 9, 184, 163, 33, 60, 55, 132, 5080, 0, 91, 73, -168, -6, 2045]
Robot Data: [9, 178, 18, 9, 183, 164, 30, 57, 54, 132, 4800, 0, 76, 67, -156, -5, 2046]
Robot Data: [8, 174, 5, 10, 181, 165, 25, 54, 50, 132, 4520, 0, 68, 69, -148, -5, 2047]
Robot Data: [13, 160, 2, 12, 177, 165, 20, 51, 47, 132, 4240, 0, 58, 66, -120, -4, 2048]
Robot Data: [13, 153, 0, 13, 172, 164, 16, 48, 44, 132, 3960, 0, 54, 57, -106, -4, 2049]
Robot Data: [16, 168, 4, 13, 171, 164, 14, 45, 41, 132, 3680, 0, 55, 43, -136, -5, 2050]
Robot Data: [19, 169, 4, 16, 171, 164, 12, 42, 38, 132, 3400, 0, 61, 32, -138, -5, 2051]
Robot Data: [22, 170, 5, 17, 171, 164, 11, 40, 35, 132, 3120, 0, 54, 27, -140, -5, 2052]
Robot Data: [27, 169, 5, 20, 171, 164, 10, 38, 32, 132, 2840, 0, 35, 35, -1, 0, 2053]
Robot Data: [30, 165, 6, 22, 170, 164, 9, 36, 30, 132, 2560, 0, 13, 37, -1, -1, 2054]
Robot Data: [34, 163, 6, 23, 169, 164, 8, 34, 27, 132, 2280, 0, 7, 31, -1, -1, 2055]
Robot Data: [36, 159, 6, 24, 167, 164, 8, 32, 23, 132, 2000, 0, 16, 24, -1, -1, 2056]
Robot Data: [41, 159, 7, 26, 165, 164, 8, 30, 19, 132, 2000, 0, 41, 27, -1, -1, 2057]
Robot Data: [44, 160, 8, 29, 164, 164, 8, 29, 20, 132, 2000, 0, 41, 20, -1, -1, 2058]
Robot Data: [48, 159, 7, 30, 163, 164, 8, 28, 19, 132, 2000, 0, 44, 24, -1, -1, 2059]
Robot Data: [55, 152, 10, 33, 161, 163, 8, 27, 19, 128, 2000, 0, 37, 33, 0, 0, 2060]
Robot Data: [54, 151, 8, 33, 159, 162, 8, 26, 18, 128, 2000, 0, 37, 42, 0, 0, 2061]
Robot Data: [60, 146, 9, 34, 156, 161, 8, 25, 17, 128, 2000, 0, 39, 50, 0, 0, 2062]
Robot Data: [60, 140, 9, 35, 153, 160, 8, 24, 19, 128, 2000, 0, 33, 43, 0, 0, 2063]
Robot Data: [63, 137, 8, 36, 150, 159, 8, 23, 17, 128, 2000, 0, 44, 50, 0, 0, 2064]
Robot Data: [65, 139, 11, 39, 148, 158, 9, 22, 17, 128, 2000, 0, 47, 50, 0, 0, 2065]
Robot Data: [71, 142, 13, 42, 147, 157, 10, 21, 18, 128, 2000, -303, 52, 40, 0, 0, 2066]
Robot Data: [70, 144, 13, 43, 146, 156, 11, 21, 18, 128, 2000, -606, 62, 32, 0, 0, 2067]
Robot Data: [74, 145, 16, 46, 146, 155, 12, 21, 18, 128, 2000, -909, 75, 21, 0, 0, 2068]
Robot Data: [76, 138, 16, 48, 144, 154, 13, 21, 17, 128, 2000, -1212, 92, 14, 0, 0, 2069]
Robot Data: [79, 126, 13, 49, 140, 152, 13, 21, 19, 128, 2000, -1515, 94, 1, 0, 0, 2070]
Robot Data: [89, 109, 10, 49, 134, 149, 12, 20, 19, 128, 2000, -1818, 95, 1, 0, 0, 2071]
Robot Data: [104, 88, 10, 48, 125, 145, 12, 19, 18, 128, 2000, -2121, 99, 5, 0, 0, 2072]
Robot Data: [112, 68, 7, 43, 114, 140, 11, 18, 19, 128, 2000, -2424, 92, 7, 0, 0, 2073]
Robot Data: [108, 77, 5, 39, 107, 136, 10, 17, 19, 128, 2000, -2727, 89, 11, 0, 0, 2074]
Robot Data: [104, 91, 2, 35, 104, 133, 8, 16, 20, 128, 2000, -3030, 87, 7, 0, 0, 2075]
Robot Data: [102, 94, 2, 30, 102, 131, 7, 15, 18, 128, 2000, -3108, 95, 14, 0, 0, 2076]
Robot Data: [101, 107, 4, 28, 103, 130, 6, 14, 20, 128, 2000, -3108, 86, 10, 0, 0, 2077]
Robot Data: [98, 125, 0, 22, 107, 130, 5, 13, 21, 128, 2000, -3108, 81, 7, 0, 0, 2078]
Robot Data: [98, 146, 0, 17, 115, 131, 4, 12, 19, 128, 2000, -3108, 80, 22, 0, 0, 2079]
Robot Data: [93, 173, 0, 11, 127, 134, 3, 11, 20, 128, 2000, -3108, 76, 20, 0, 0, 2080]
Robot Data: [85, 212, 1, 8, 144, 139, 3, 10, 20, 128, 2000, -3108, 78, 17, 0, 0, 2081]
Robot Data: [77, 229, 0, 4, 161, 145, 2, 9, 21, 128, 2000, -3108, 80, 8, 0, 0, 2082]
Robot Data: [69, 205, 2, 4, 170, 149, 2, 9, 20, 128, 2000, -3108, 82, 12, 0, 0, 2083]
Robot Data: [58, 178, 4, 1, 172, 151, 2, 9, 19, 128, 2000, -3108, 86, 16, 0, 0, 2084]
Robot Data: [45, 159, 10, 1, 169, 152, 4, 9, 20, 128, 2000, -3108, 81, 15, 0, 0, 2085]
Robot Data: [34, 147, 20, 1, 165, 152, 7, 10, 20, 128, 2000, -3108, 83, 12, 0, 0, 2086]
Robot Data: [25, 129, 38, 1, 158, 151, 13, 12, 20, 128, 2000, -3108, 83, 13, 0, 0, 2087]
Robot Data: [16, 102, 68, 0, 147, 148, 24, 16, 21, 128, 2000, -2805, 75, 13, 0, 0, 2088]
Robot Data: [4, 98, 109, 0, 137, 145, 41, 22, 20, 128, 2000, -2502, 67, 27, 0, 0, 2089]
Robot Data: [3, 85, 158, 1, 127, 141, 64, 31, 20, 128, 2000, -2199, 55, 39, 0, 0, 2090]
Robot Data: [0, 70, 211, 1, 116, 137, 93, 42, 20, 128, 2000, -1896, 47, 47, 0, 0, 2091]
Robot Data: [1, 56, 258, 1, 104, 132, 126, 56, 20, 128, 2000, -1593, 43, 51, 0, 0, 2092]
Robot Data: [1, 46, 301, 2, 92, 127, 161, 71, 20, 128, 2000, -1290, 34, 60, 0, 0, 2093]
Robot Data: [0, 42, 333, 4, 82, 122, 195, 87, 20, 128, 2000, -987, 34, 61, 0, 0, 2094]
Robot Data: [0, 39, 354, 4, 73, 117, 227, 104, 20, 128, 2000, -684, 36, 58, 0, 0, 2095]
Robot Data: [1, 37, 363, 5, 66, 112, 254, 120, 20, 128, 2000, -381, 34, 61, 0, 0, 2096]
Robot Data: [1, 37, 370, 5, 60, 107, 277, 136, 19, 128, 2000, -78, 40, 62, 0, 0, 2097]
Robot Data: [1, 37, 366, 4, 55, 103, 295, 150, 20, 128, 2000, 0, 42, 54, 0, 0, 2098]
Robot Data: [0, 39, 361, 5, 52, 99, 308, 163, 19, 112, 2280, 0, 31, 95, 255, 16, 2099]
Robot Data: [0, 40, 350, 3, 50, 95, 316, 175, 21, 112, 2560, 0, 30, 114, 255, 14, 2100]
Robot Data: [2, 44, 327, 2, 49, 92, 318, 185, 21, 112, 2840, 0, 62, 118, 255, 7, 2101]
Robot Data: [1, 53, 289, 2, 50, 90, 312, 192, 24, 112, 3120, 0, 85, 110, 236, 6, 2102]
Robot Data: [0, 62, 242, 1, 52, 88, 298, 195, 28, 112, 3400, 0, 102, 101, 180, 5, 2103]
Robot Data: [1, 71, 210, 0, 56, 87, 280, 196, 30, 112, 3680, 0, 124, 92, 139, 4, 2104]
Robot Data: [3, 77, 177, 1, 60, 86, 259, 195, 32, 112, 3960, 0, 146, 92, 100, 2, 2105]
Robot Data: [1, 85, 156, 1, 65, 86, 238, 193, 36, 112, 4240, 0, 148, 98, 71, 2, 2106]
Robot Data: [0, 88, 145, 1, 70, 86, 219, 190, 38, 112, 4520, 0, 153, 114, 57, 1, 2107]
Robot Data: [2, 90, 139, 1, 74, 86, 203, 187, 43, 112, 4800, 0, 140, 127, 49, 1, 2108]
Robot Data: [0, 91, 138, 0, 77, 86, 190, 184, 44, 112, 5080, 0, 140, 147, 47, 1, 2109]
Robot Data: [0, 92, 137, 0, 80, 86, 179, 181, 47, 112, 5360, 0, 141, 160, 45, 1, 2110]
Robot Data: [0, 90, 133, 1, 82, 86, 170, 178, 52, 112, 5640, 0, 139, 160, 43, 1, 2111]
Robot Data: [0, 89, 136, 0, 83, 86, 163, 175, 56, 112, 5920, 0, 150, 154, 47, 1, 2112]
Robot Data: [1, 89, 141, 1, 84, 86, 159, 173, 57, 112, 6200, 0, 173, 157, 52, 1, 2113]
Robot Data: [1, 88, 140, 0, 85, 86, 155, 171, 55, 112, 6480, 0, 194, 180, 52, 1, 2114]
Robot Data: [1, 91, 141, 1, 86, 86, 152, 169, 59, 112, 6760, 0, 191, 193, 50, 1, 2115]
Robot Data: [0, 83, 132, 1, 85, 86, 148, 167, 64, 112, 7040, 0, 184, 200, 49, 1, 2116]
Robot Data: [1, 84, 126, 0, 85, 86, 144, 164, 68, 112, 7320, 0, 184, 206, 42, 1, 2117]
Robot Data: [0, 85, 131, 0, 85, 86, 141, 162, 71, 112, 7600, 0, 199, 204, 46, 1, 2118]
Robot Data: [1, 90, 132, 1, 86, 86, 139, 160, 74, 112, 7880, 0, 205, 203, 42, 1, 2119]
Robot Data: [1, 92, 133, 0, 87, 86, 138, 158, 73, 112, 8160, 0, 227, 224, 41, 1, 2120]
Robot Data: [0, 93, 126, 0, 88, 86, 136, 156, 81, 112, 8440, 0, 216, 212, 33, 0, 2121]
Robot Data: [0, 95, 124, 0, 89, 87, 134, 154, 80, 112, 8720, 0, 237, 233, 29, 0, 2122]
Robot Data: [1, 95, 125, 1, 90, 88, 132, 152, 86, 112, 9000, 0, 233, 229, 30, 0, 2123]
Robot Data: [2, 95, 125, 0, 91, 88, 131, 150, 90, 112, 9000, 0, 221, 214, 30, 0, 2124]
Robot Data: [0, 93, 127, 1, 91, 88, 130, 149, 90, 112, 9000, 0, 225, 210, 34, 1, 2125]
Robot Data: [1, 88, 130, 0, 90, 88, 130, 148, 89, 112, 9000, 0, 228, 215, 42, 1, 2126]
Robot Data: [2, 86, 135, 1, 89, 88, 131, 147, 89, 112, 9000, 0, 225, 219, 49, 1, 2127]
Robot Data: [1, 84, 137, 1, 88, 88, 132, 146, 91, 112, 9000, 0, 210, 219, 53, 1, 2128]
Robot Data: [1, 78, 138, 1, 86, 87, 133, 146, 90, 112, 9000, 0, 209, 227, 60, 1, 2129]
Robot Data: [1, 89, 149, 1, 87, 87, 136, 146, 93, 112, 9000, 0, 200, 213, 60, 1, 2130]
Robot Data: [1, 85, 163, 2, 87, 87, 141, 147, 92, 112, 9000, 0, 204, 213, 78, 2, 2131]
Robot Data: [1, 84, 166, 1, 86, 87, 146, 148, 92, 112, 9000, 0, 204, 211, 82, 2, 2132]
Robot Data: [1, 84, 168, 2, 86, 87, 150, 149, 92, 112, 9000, 0, 205, 208, 84, 2, 2133]
Robot Data: [1, 84, 167, 3, 86, 87, 153, 150, 93, 112, 9000, 0, 200, 203, 83, 2, 2134]
Robot Data: [2, 85, 166, 1, 86, 87, 156, 151, 93, 112, 9000, 0, 194, 205, 81, 2, 2135]
Robot Data: [1, 84, 162, 1, 86, 87, 157, 152, 92, 112, 9000, 0, 197, 207, 78, 2, 2136]
Robot Data: [2, 87, 160, 1, 86, 87, 158, 153, 93, 112, 9000, 0, 194, 200, 73, 2, 2137]
Robot Data: [2, 88, 160, 2, 86, 87, 158, 153, 93, 112, 8720, 0, 177, 183, 72, 2, 2138]
Robot Data: [1, 93, 154, 2, 87, 87, 157, 153, 92, 112, 8440, 0, 164, 167, 61, 1, 2139]
Robot Data: [2, 91, 143, 3, 88, 87, 154, 152, 89, 112, 8160, 0, 157, 157, 52, 1, 2140]
Robot Data: [2, 85, 142, 3, 87, 87, 152, 151, 86, 112, 7880, 0, 153, 145, 57, 1, 2141]
Robot Data: [1, 86, 139, 2, 87, 87, 149, 150, 83, 112, 7600, 0, 148, 141, 53, 1, 2142]
Robot Data: [3, 93, 137, 4, 88, 87, 147, 149, 82, 112, 7320, 0, 131, 128, 44, 1, 2143]
Robot Data: [2, 95, 133, 4, 89, 88, 144, 148, 79, 112, 7040, 0, 121, 121, 38, 1, 2144]
Robot Data: [4, 94, 132, 4, 90, 88, 142, 147, 76, 112, 6760, 0, 114, 110, 38, 1, 2145]
Robot Data: [3, 97, 130, 4, 91, 89, 140, 146, 72, 112, 6480, 0, 111, 104, 33, 0, 2146]
Robot Data: [4, 97, 131, 4, 92, 90, 138, 145, 68, 112, 6200, 0, 108, 106, 34, 1, 2147]
Robot Data: [4, 99, 129, 6, 93, 91, 136, 144, 66, 112, 5920, 0, 96, 96, 30, 0, 2148]
Robot Data: [5, 99, 126, 7, 94, 92, 134, 143, 63, 112, 5640, 0, 87, 91, 27, 0, 2149]
Robot Data: [6, 99, 124, 8, 95, 92, 132, 142, 59, 112, 5360, 0, 83, 87, 25, 0, 2150]
Robot Data: [7, 101, 125, 9, 96, 93, 131, 141, 56, 112, 5080, 0, 80, 76, 24, 0, 2151]
Robot Data: [7, 104, 121, 10, 98, 94, 129, 140, 52, 112, 4800, 0, 80, 77, 17, 0, 2152]
Robot Data: [8, 102, 116, 11, 99, 95, 126, 139, 51, 112, 4520, 0, 69, 62, 14, 0, 2153]
Robot Data: [10, 101, 108, 11, 99, 95, 122, 137, 47, 112, 4240, 0, 65, 59, 7, 0, 2154]
Robot Data: [11, 97, 61, 12, 99, 95, 110, 132, 45, 112, 3960, 0, 59, 45, -36, -2, 2155]
Robot Data: [14, 99, 14, 13, 99, 95, 91, 125, 41, 112, 3680, 0, 51, 46, 2, 0, 2156]
Robot Data: [15, 105, 6, 15, 100, 96, 74, 118, 37, 112, 3400, 0, 49, 51, -10, -1, 2157]
Robot Data: [18, 106, 4, 16, 101, 97, 60, 111, 35, 112, 3120, 0, 37, 45, -12, -1, 2158]
Robot Data: [22, 106, 5, 18, 102, 98, 49, 104, 33, 112, 2840, 0, 30, 32, -12, -1, 2159]
Robot Data: [23, 107, 5, 19, 103, 99, 40, 98, 30, 112, 2560, 0, 25, 24, -1, -1, 2160]
Robot Data: [24, 109, 2, 21, 104, 100, 32, 92, 27, 112, 2280, 0, 20, 16, -1, -1, 2161]
Robot Data: [28, 110, 4, 23, 105, 101, 26, 87, 23, 112, 2000, 0, 21, 18, -1, -1, 2162]
Robot Data: [29, 109, 6, 24, 106, 102, 22, 82, 20, 112, 2000, 0, 33, 26, -1, -1, 2163]
Robot Data: [30, 107, 5, 25, 106, 102, 19, 77, 19, 112, 2000, 0, 37, 30, -1, -1, 2164]
Robot Data: [31, 104, 5, 25, 106, 102, 16, 73, 19, 112, 2000, 0, 35, 33, -1, -1, 2165]
Robot Data: [34, 105, 5, 27, 106, 102, 14, 69, 17, 48, 2000, 0, 40, 44, 0, 0, 2166]
Robot Data: [34, 102, 5, 27, 105, 102, 12, 65, 18, 48, 2000, 0, 38, 42, 0, 0, 2167]
Robot Data: [36, 104, 6, 28, 105, 102, 11, 61, 17, 48, 2000, 0, 45, 45, 0, 0, 2168]
Robot Data: [37, 101, 6, 28, 104, 102, 10, 58, 17, 48, 2000, 0, 49, 45, 0, 0, 2169]
Robot Data: [37, 105, 5, 30, 104, 102, 9, 55, 19, 48, 2000, 0, 38, 43, 0, 0, 2170]
Robot Data: [39, 103, 5, 32, 104, 102, 8, 52, 17, 48, 2000, 0, 47, 51, 0, 0, 2171]
Robot Data: [39, 102, 5, 34, 104, 102, 7, 49, 17, 48, 2000, 0, 51, 50, 0, 0, 2172]
Robot Data: [41, 104, 4, 35, 104, 102, 6, 46, 18, 48, 2000, 0, 50, 46, 0, 0, 2173]
Robot Data: [43, 105, 5, 38, 104, 102, 6, 43, 18, 48, 2000, 0, 49, 50, 0, 0, 2174]
Robot Data: [47, 107, 6, 40, 105, 102, 6, 41, 19, 48, 2000, 0, 47, 47, 0, 0, 2175]
Robot Data: [45, 106, 4, 42, 105, 102, 6, 39, 19, 48, 2000, 0, 51, 43, 0, 0, 2176]
Robot Data: [48, 106, 5, 44, 105, 102, 6, 37, 19, 48, 2000, 0, 49, 46, 0, 0, 2177]
Robot Data: [50, 105, 4, 46, 105, 102, 6, 35, 19, 48, 2000, 0, 48, 49, 0, 0, 2178]
Robot Data: [55, 107, 4, 48, 105, 102, 6, 33, 20, 48, 2000, 0, 45, 45, 0, 0, 2179]
Robot Data: [58, 109, 4, 51, 106, 102, 6, 31, 19, 48, 2000, -303, 55, 43, 0, 0, 2180]
Robot Data: [58, 109, 5, 53, 107, 102, 6, 29, 19, 48, 2000, -606, 64, 35, 0, 0, 2181]
Robot Data: [60, 107, 4, 56, 107, 102, 6, 27, 19, 48, 2000, -909, 75, 25, 0, 0, 2182]
Robot Data: [64, 104, 4, 58, 106, 102, 6, 26, 20, 48, 2000, -1212, 82, 11, 0, 0, 2183]
Robot Data: [67, 99, 3, 57, 105, 102, 5, 25, 19, 48, 2000, -1515, 92, 9, 0, 0, 2184]
Robot Data: [77, 90, 6, 57, 102, 101, 5, 24, 19, 48, 2000, -1818, 97, 5, 0, 0, 2185]
Robot Data: [91, 76, 7, 53, 97, 99, 5, 23, 20, 48, 2000, -2121, 95, 1, 0, 0, 2186]
Robot Data: [112, 65, 8, 50, 91, 97, 6, 22, 20, 48, 2000, -2424, 94, 1, 0, 0, 2187]
Robot Data: [126, 78, 9, 44, 88, 96, 7, 21, 19, 48, 2000, -2727, 94, 9, 0, 0, 2188]
Robot Data: [120, 76, 10, 32, 86, 95, 8, 20, 20, 48, 2000, -3030, 92, 5, 0, 0, 2189]
Robot Data: [111, 89, 13, 8, 87, 95, 9, 20, 20, 48, 2000, -3108, 90, 7, 0, 0, 2190]
Robot Data: [109, 105, 17, 5, 91, 96, 11, 20, 20, 48, 2000, -3108, 84, 12, 0, 0, 2191]
Robot Data: [109, 126, 18, 4, 98, 98, 12, 20, 20, 48, 2000, -3108, 79, 17, 0, 0, 2192]
Robot Data: [109, 148, 23, 4, 108, 101, 14, 20, 20, 48, 2000, -3108, 78, 18, 0, 0, 2193]
Robot Data: [103, 171, 29, 3, 121, 105, 17, 21, 21, 48, 2000, -3108, 77, 12, 0, 0, 2194]
Robot Data: [100, 207, 37, 4, 138, 111, 21, 22, 20, 48, 2000, -3108, 80, 16, 0, 0, 2195]
Robot Data: [91, 259, 33, 6, 162, 120, 23, 23, 20, 48, 2000, -3108, 80, 16, 0, 0, 2196]
Robot Data: [84, 274, 31, 9, 184, 130, 25, 24, 20, 48, 2000, -3108, 80, 16, 0, 0, 2197]
Robot Data: [69, 229, 29, 9, 193, 136, 26, 24, 20, 48, 2000, -3108, 82, 14, 0, 0, 2198]
Robot Data: [20, 192, 28, 12, 193, 140, 26, 24, 20, 48, 2000, -3108, 85, 10, 0, 0, 2199]
Robot Data: [9, 171, 19, 12, 189, 142, 25, 24, 20, 48, 2000, -3108, 85, 11, 0, 0, 2200]
Robot Data: [7, 139, 18, 15, 179, 142, 24, 24, 19, 48, 2000, -3108, 85, 19, 0, 0, 2201]
Robot Data: [9, 62, 29, 17, 156, 137, 25, 24, 21, 48, 2000, -2805, 74, 15, 0, 0, 2202]
Robot Data: [13, 10, 54, 21, 127, 129, 31, 26, 20, 48, 2000, -2502, 70, 26, 0, 0, 2203]
Robot Data: [15, 7, 88, 27, 103, 121, 42, 30, 21, 48, 2000, -2199, 53, 34, 0, 0, 2204]
Robot Data: [18, 7, 128, 31, 84, 114, 59, 36, 20, 48, 2000, -1896, 49, 45, 0, 0, 2205]
Robot Data: [21, 7, 169, 34, 69, 107, 81, 44, 19, 48, 2000, -1593, 47, 55, 0, 0, 2206]
Robot Data: [25, 6, 204, 35, 56, 101, 106, 54, 21, 48, 2000, -1290, 33, 55, 0, 0, 2207]
Robot Data: [30, 7, 231, 37, 46, 95, 131, 65, 20, 48, 2000, -987, 31, 63, 0, 0, 2208]
Robot Data: [33, 7, 242, 39, 38, 90, 153, 76, 19, 48, 2000, -684, 38, 64, 0, 0, 2209]
Robot Data: [34, 8, 248, 39, 32, 85, 172, 87, 20, 48, 2000, -381, 34, 61, 0, 0, 2210]
Robot Data: [36, 8, 252, 41, 27, 80, 188, 97, 20, 48, 2000, -78, 37, 59, 0, 0, 2211]
Robot Data: [34, 8, 246, 42, 23, 76, 200, 106, 19, 48, 2000, 0, 44, 59, 0, 0, 2212]
Robot Data: [40, 10, 239, 47, 20, 72, 208, 114, 21, 49, 2000, 0, 40, 49, 0, 0, 2213]
Robot Data: [38, 9, 232, 47, 18, 68, 213, 121, 20, 49, 2000, 0, 51, 45, 0, 0, 2214]
Robot Data: [41, 9, 223, 49, 16, 64, 215, 127, 21, 49, 2000, 0, 49, 38, 0, 0, 2215]
Robot Data: [43, 9, 215, 52, 15, 61, 215, 133, 20, 49, 2000, 0, 52, 42, 0, 0, 2216]
Robot Data: [44, 10, 208, 54, 14, 58, 214, 138, 19, 49, 2000, 0, 60, 42, 0, 0, 2217]
Robot Data: [46, 11, 204, 56, 13, 55, 212, 142, 20, 49, 2000, 0, 52, 43, 0, 0, 2218]
Robot Data: [52, 11, 204, 58, 13, 52, 210, 146, 21, 49, 2000, 303, 43, 45, 0, 0, 2219]
Robot Data: [54, 10, 208, 62, 12, 49, 210, 150, 19, 49, 2000, 606, 37, 65, 0, 0, 2220]
Robot Data: [55, 12, 209, 63, 12, 47, 210, 154, 20, 49, 2000, 909, 27, 69, 0, 0, 2221]
Robot Data: [58, 13, 208, 67, 12, 45, 210, 157, 20, 49, 2000, 1212, 12, 84, 0, 0, 2222]
Robot Data: [60, 13, 193, 73, 12, 43, 207, 159, 19, 49, 2000, 1515, 11, 93, 0, 0, 2223]
Robot Data: [64, 12, 170, 80, 12, 41, 200, 160, 20, 49, 2000, 1818, 1, 98, 0, 0, 2224]
Robot Data: [58, 12, 140, 86, 12, 39, 188, 159, 19, 49, 2000, 2121, 0, 105, 0, 0, 2225]
Robot Data: [55, 8, 105, 87, 11, 37, 171, 156, 20, 49, 2000, 2424, 1, 99, 0, 0, 2226]
Robot Data: [52, 8, 99, 85, 10, 35, 157, 152, 20, 49, 2000, 2727, 1, 97, 0, 0, 2227]
Robot Data: [46, 8, 133, 82, 10, 33, 152, 151, 19, 49, 2000, 3030, 10, 95, 0, 0, 2228]
Robot Data: [42, 12, 129, 83, 10, 32, 147, 150, 20, 49, 2000, 3108, 12, 87, 0, 0, 2229]
Robot Data: [34, 13, 155, 81, 11, 31, 149, 150, 20, 49, 2000, 3108, 15, 84, 0, 0, 2230]
Robot Data: [25, 14, 178, 80, 12, 30, 155, 152, 20, 49, 2000, 3108, 20, 79, 0, 0, 2231]
Robot Data: [20, 17, 203, 78, 13, 29, 165, 155, 20, 49, 2000, 3108, 21, 78, 0, 0, 2232]
Robot Data: [14, 22, 233, 74, 15, 29, 179, 160, 20, 49, 2000, 3108, 21, 78, 0, 0, 2233]
Robot Data: [10, 23, 291, 67, 17, 29, 201, 168, 21, 49, 2000, 3108, 11, 80, 0, 0, 2234]
Robot Data: [10, 21, 326, 60, 18, 29, 226, 178, 20, 49, 2000, 3108, 15, 82, 0, 0, 2235]
Robot Data: [15, 18, 308, 52, 18, 28, 242, 186, 20, 49, 2000, 3108, 17, 81, 0, 0, 2236]
Robot Data: [15, 15, 273, 47, 17, 27, 248, 191, 20, 49, 2000, 3108, 15, 83, 0, 0, 2237]
Robot Data: [17, 9, 239, 37, 15, 26, 246, 194, 20, 49, 2000, 3108, 15, 82, 0, 0, 2238]
Robot Data: [19, 2, 225, 28, 12, 25, 242, 196, 21, 49, 2000, 3108, 10, 81, 0, 0, 2239]
Robot Data: [21, 1, 201, 19, 10, 24, 234, 196, 21, 49, 2000, 3108, 7, 82, 0, 0, 2240]
Robot Data: [27, 4, 175, 13, 9, 23, 222, 195, 20, 49, 2000, 2805, 19, 77, 0, 0, 2241]
Robot Data: [30, 2, 155, 11, 8, 22, 209, 193, 20, 49, 2000, 2502, 27, 68, 0, 0, 2242]
Robot Data: [34, 1, 132, 19, 7, 21, 194, 189, 21, 49, 2000, 2199, 36, 52, 0, 0, 2243]
Robot Data: [29, 2, 108, 18, 6, 20, 177, 184, 20, 49, 2000, 1896, 49, 46, 0, 0, 2244]
Robot Data: [21, 2, 89, 20, 5, 19, 159, 178, 19, 49, 2000, 1593, 61, 42, 0, 0, 2245]
Robot Data: [13, 2, 75, 22, 4, 18, 142, 172, 20, 49, 2000, 1290, 62, 33, 0, 0, 2246]
Robot Data: [7, 2, 67, 25, 4, 17, 127, 165, 20, 49, 2000, 987, 67, 29, 0, 0, 2247]
Robot Data: [7, 3, 62, 27, 4, 16, 114, 159, 19, 49, 2000, 684, 68, 36, 0, 0, 2248]
Robot Data: [4, 2, 58, 28, 4, 15, 103, 153, 20, 49, 2000, 381, 63, 34, 0, 0, 2249]
Robot Data: [6, 3, 58, 30, 4, 14, 94, 147, 20, 49, 2000, 78, 59, 37, 0, 0, 2250]
Robot Data: [8, 3, 59, 31, 4, 13, 87, 142, 20, 49, 2000, 0, 56, 41, 0, 0, 2251]
Robot Data: [12, 2, 62, 33, 4, 12, 82, 137, 20, 33, 2000, 0, 49, 48, 0, 0, 2252]
Robot Data: [15, 1, 64, 33, 3, 11, 78, 132, 20, 33, 2000, 0, 42, 54, 0, 0, 2253]
Robot Data: [16, 2, 68, 33, 3, 10, 76, 128, 21, 33, 2000, 0, 39, 49, 0, 0, 2254]
Robot Data: [21, 3, 69, 35, 3, 10, 75, 124, 20, 33, 2000, 0, 43, 53, 0, 0, 2255]
Robot Data: [26, 3, 70, 37, 3, 10, 74, 121, 19, 33, 2000, 0, 47, 57, 0, 0, 2256]
Robot Data: [27, 3, 70, 39, 3, 10, 73, 118, 20, 33, 2000, 0, 45, 51, 0, 0, 2257]
Robot Data: [28, 2, 69, 40, 3, 10, 72, 115, 20, 33, 2000, 0, 47, 50, 0, 0, 2258]
Robot Data: [32, 3, 69, 44, 3, 10, 71, 112, 20, 33, 2000, 0, 49, 48, 0, 0, 2259]
Robot Data: [31, 3, 66, 45, 3, 10, 70, 109, 20, 33, 2000, 0, 50, 46, 0, 0, 2260]
Robot Data: [33, 3, 66, 49, 3, 10, 69, 106, 20, 33, 2000, 0, 52, 45, 0, 0, 2261]
Robot Data: [34, 3, 63, 51, 3, 10, 68, 103, 20, 33, 2000, 0, 50, 47, 0, 0, 2262]
Robot Data: [36, 3, 64, 54, 3, 10, 67, 101, 20, 33, 2000, 0, 54, 43, 0, 0, 2263]
Robot Data: [41, 3, 66, 58, 3, 10, 67, 99, 21, 33, 2000, 0, 46, 43, 0, 0, 2264]
Robot Data: [45, 3, 66, 60, 3, 10, 67, 97, 20, 33, 2000, 0, 49, 46, 0, 0, 2265]
Robot Data: [52, 3, 69, 64, 3, 10, 67, 95, 19, 33, 2000, 303, 46, 58, 0, 0, 2266]
Robot Data: [56, 4, 70, 66, 3, 10, 68, 93, 21, 33, 2000, 606, 28, 61, 0, 0, 2267]
Robot Data: [61, 3, 75, 72, 3, 10, 69, 92, 20, 33, 2000, 909, 19, 76, 0, 0, 2268]
Robot Data: [62, 3, 78, 77, 3, 10, 71, 91, 19, 33, 2000, 1212, 17, 86, 0, 0, 2269]
Robot Data: [58, 3, 89, 85, 3, 10, 75, 91, 20, 33, 2000, 1515, 9, 88, 0, 0, 2270]
Robot Data: [46, 3, 92, 94, 3, 10, 78, 91, 20, 33, 2000, 1818, 4, 92, 0, 0, 2271]
Robot Data: [22, 4, 84, 107, 3, 10, 79, 91, 20, 33, 2000, 2121, 3, 94, 0, 0, 2272]
Robot Data: [6, 5, 92, 113, 3, 10, 82, 91, 19, 33, 2000, 2424, 5, 99, 0, 0, 2273]
Robot Data: [3, 4, 117, 109, 3, 10, 89, 93, 20, 33, 2000, 2727, 4, 93, 0, 0, 2274]
Robot Data: [4, 6, 145, 106, 4, 10, 100, 96, 20, 33, 2000, 3030, 4, 94, 0, 0, 2275]
Robot Data: [4, 9, 181, 109, 5, 10, 116, 101, 19, 33, 2000, 3108, 11, 94, 0, 0, 2276]
Robot Data: [2, 10, 225, 110, 6, 10, 138, 109, 20, 33, 2000, 3108, 10, 89, 0, 0, 2277]
Robot Data: [2, 15, 262, 111, 8, 10, 163, 119, 20, 33, 2000, 3108, 15, 84, 0, 0, 2278]
Robot Data: [2, 19, 299, 109, 10, 11, 190, 130, 20, 33, 2000, 3108, 20, 79, 0, 0, 2279]
Robot Data: [5, 60, 348, 105, 20, 14, 222, 144, 21, 33, 2000, 3108, 17, 74, 0, 0, 2280]
Robot Data: [7, 80, 410, 92, 32, 18, 260, 161, 20, 33, 2000, 3108, 18, 79, 0, 0, 2281]
Robot Data: [8, 21, 453, 49, 30, 18, 299, 179, 21, 33, 2000, 3108, 13, 77, 0, 0, 2282]
Robot Data: [10, 19, 441, 8, 28, 18, 327, 195, 21, 33, 2000, 3108, 11, 78, 0, 0, 2283]
Robot Data: [18, 19, 337, 3, 26, 18, 329, 204, 19, 33, 2000, 3108, 18, 85, 0, 0, 2284]
Robot Data: [17, 26, 211, 4, 26, 19, 305, 204, 20, 33, 2000, 3108, 13, 84, 0, 0, 2285]
Robot Data: [18, 30, 18, 6, 27, 20, 248, 192, 20, 33, 2000, 3108, 15, 82, 0, 0, 2286]
Robot Data: [23, 22, 10, 7, 26, 20, 200, 181, 20, 33, 2000, 3108, 15, 82, 0, 0, 2287]
Robot Data: [31, 23, 8, 9, 25, 20, 162, 170, 20, 33, 2000, 2805, 17, 80, 0, 0, 2288]
Robot Data: [41, 36, 7, 11, 27, 21, 131, 160, 20, 33, 2000, 2502, 24, 72, 0, 0, 2289]
Robot Data: [49, 57, 7, 13, 33, 23, 106, 150, 20, 33, 2000, 2199, 39, 58, 0, 0, 2290]
Robot Data: [46, 78, 6, 20, 42, 26, 86, 141, 21, 33, 2000, 1896, 43, 46, 0, 0, 2291]
Robot Data: [39, 102, 4, 23, 54, 31, 70, 132, 20, 33, 2000, 1593, 54, 42, 0, 0, 2292]
Robot Data: [42, 123, 7, 30, 68, 37, 57, 124, 19, 33, 2000, 1290, 67, 37, 0, 0, 2293]
Robot Data: [44, 142, 9, 39, 83, 44, 47, 117, 20, 33, 2000, 987, 68, 29, 0, 0, 2294]
Robot Data: [44, 154, 10, 44, 97, 51, 40, 110, 20, 33, 2000, 684, 68, 28, 0, 0, 2295]
Robot Data: [40, 159, 12, 47, 109, 58, 34, 104, 20, 33, 2000, 381, 67, 30, 0, 0, 2296]
Robot Data: [47, 162, 14, 49, 120, 65, 30, 98, 20, 33, 2000, 78, 63, 34, 0, 0, 2297]
Robot Data: [53, 157, 15, 50, 127, 71, 27, 93, 20, 33, 2000, 0, 58, 39, 0, 0, 2298]
Robot Data: [59, 150, 15, 51, 132, 76, 25, 88, 19, 32, 2000, 0, 53, 52, 0, 0, 2299]
Robot Data: [63, 148, 15, 54, 135, 81, 23, 83, 20, 32, 2000, -303, 50, 47, 0, 0, 2300]
Robot Data: [61, 140, 16, 55, 136, 85, 22, 79, 21, 32, 2000, -606, 53, 37, 0, 0, 2301]
Robot Data: [61, 136, 17, 55, 136, 88, 21, 75, 20, 32, 2000, -909, 67, 30, 0, 0, 2302]
Robot Data: [65, 132, 17, 60, 135, 91, 20, 71, 20, 32, 2000, -1212, 77, 20, 0, 0, 2303]
Robot Data: [71, 122, 19, 60, 132, 93, 20, 68, 19, 32, 2000, -1515, 87, 18, 0, 0, 2304]
Robot Data: [75, 109, 17, 55, 127, 94, 19, 65, 20, 32, 2000, -1818, 92, 6, 0, 0, 2305]
Robot Data: [90, 95, 14, 52, 121, 94, 18, 62, 20, 32, 2000, -2121, 100, 3, 0, 0, 2306]
Robot Data: [116, 77, 11, 52, 112, 93, 17, 59, 20, 32, 2000, -2424, 101, 3, 0, 0, 2307]
Robot Data: [135, 91, 8, 49, 108, 93, 15, 56, 20, 32, 2000, -2727, 98, 1, 0, 0, 2308]
Robot Data: [127, 86, 7, 46, 104, 93, 13, 53, 20, 32, 2000, -3030, 99, 1, 0, 0, 2309]
Robot Data: [117, 94, 8, 41, 102, 93, 12, 50, 19, 32, 2000, -3108, 95, 10, 0, 0, 2310]
Robot Data: [114, 113, 13, 36, 104, 94, 12, 48, 20, 32, 2000, -3108, 85, 14, 0, 0, 2311]
Robot Data: [116, 138, 13, 28, 111, 97, 12, 46, 19, 32, 2000, -3108, 86, 20, 0, 0, 2312]
Robot Data: [112, 159, 15, 21, 121, 101, 13, 44, 21, 32, 2000, -3108, 76, 16, 0, 0, 2313]
Robot Data: [113, 182, 22, 16, 133, 106, 15, 43, 21, 32, 2000, -3108, 74, 17, 0, 0, 2314]
Robot Data: [109, 222, 33, 11, 151, 113, 19, 42, 20, 32, 2000, -3108, 74, 24, 0, 0, 2315]
Robot Data: [106, 351, 21, 8, 191, 128, 19, 41, 20, 32, 2000, -3108, 78, 20, 0, 0, 2316]
Robot Data: [96, 338, 18, 7, 220, 141, 19, 40, 20, 32, 2000, -3108, 81, 16, 0, 0, 2317]
Robot Data: [88, 281, 20, 8, 232, 150, 19, 39, 20, 32, 2000, -3108, 89, 9, 0, 0, 2318]
Robot Data: [79, 232, 21, 11, 232, 155, 19, 38, 20, 32, 2000, -3108, 87, 10, 0, 0, 2319]
Robot Data: [72, 204, 24, 10, 226, 158, 20, 37, 20, 32, 2000, -3108, 86, 12, 0, 0, 2320]
Robot Data: [52, 189, 17, 12, 219, 160, 19, 36, 21, 32, 2000, -3108, 84, 6, 0, 0, 2321]
Robot Data: [36, 168, 19, 12, 209, 161, 19, 35, 20, 32, 2000, -2805, 81, 16, 0, 0, 2322]
Robot Data: [20, 156, 35, 13, 198, 161, 22, 35, 20, 32, 2000, -2502, 66, 31, 0, 0, 2323]
Robot Data: [11, 127, 61, 15, 184, 159, 30, 37, 21, 32, 2000, -2199, 53, 36, 0, 0, 2324]
Robot Data: [8, 107, 89, 15, 169, 156, 42, 40, 20, 32, 2000, -1896, 48, 48, 0, 0, 2325]
Robot Data: [13, 90, 116, 18, 153, 152, 57, 45, 19, 32, 2000, -1593, 46, 58, 0, 0, 2326]
Robot Data: [19, 79, 142, 21, 138, 147, 74, 51, 20, 32, 2000, -1290, 33, 63, 0, 0, 2327]
Robot Data: [22, 70, 159, 22, 124, 142, 91, 58, 21, 32, 2000, -987, 29, 60, 0, 0, 2328]
Robot Data: [24, 67, 174, 24, 113, 137, 108, 65, 20, 32, 2000, -684, 35, 61, 0, 0, 2329]
Robot Data: [25, 65, 178, 25, 103, 133, 122, 72, 20, 32, 2000, -381, 36, 59, 0, 0, 2330]
Robot Data: [22, 66, 179, 26, 96, 129, 133, 79, 20, 32, 2000, -78, 36, 59, 0, 0, 2331]
Robot Data: [25, 65, 174, 26, 90, 125, 141, 85, 20, 32, 2000, 0, 38, 58, 0, 0, 2332]
Robot Data: [28, 71, 167, 28, 86, 122, 146, 90, 20, 16, 2000, 0, 48, 48, 0, 0, 2333]
Robot Data: [32, 74, 161, 30, 84, 119, 149, 94, 19, 16, 2000, 0, 54, 49, 0, 0, 2334]
Robot Data: [28, 76, 153, 29, 82, 116, 150, 98, 20, 16, 2000, 0, 52, 45, 0, 0, 2335]
Robot Data: [32, 78, 147, 31, 81, 114, 149, 101, 20, 16, 2000, 0, 54, 43, 0, 0, 2336]
Robot Data: [33, 81, 147, 32, 81, 112, 149, 104, 20, 16, 2000, 0, 57, 39, 0, 0, 2337]
Robot Data: [35, 81, 144, 33, 81, 110, 148, 107, 19, 16, 2000, 0, 59, 46, 0, 0, 2338]
Robot Data: [37, 80, 142, 35, 81, 108, 147, 109, 20, 16, 2000, 0, 52, 46, 0, 0, 2339]
Robot Data: [36, 80, 144, 35, 81, 106, 146, 111, 20, 16, 2000, 0, 50, 48, 0, 0, 2340]
Robot Data: [39, 80, 145, 37, 81, 104, 146, 113, 20, 16, 2000, 0, 49, 49, 0, 0, 2341]
Robot Data: [43, 81, 148, 38, 81, 103, 146, 115, 21, 16, 2000, 0, 45, 45, 0, 0, 2342]
Robot Data: [43, 78, 148, 40, 80, 101, 146, 117, 20, 16, 2000, 0, 44, 52, 0, 0, 2343]
Robot Data: [47, 78, 149, 43, 80, 100, 147, 119, 20, 16, 2000, 0, 45, 52, 0, 0, 2344]
Robot Data: [47, 79, 148, 46, 80, 99, 147, 121, 20, 16, 2000, 0, 47, 50, 0, 0, 2345]
Robot Data: [50, 79, 142, 48, 80, 98, 146, 122, 20, 16, 2000, 0, 47, 50, 0, 0, 2346]
Robot Data: [55, 81, 143, 51, 80, 97, 145, 123, 21, 16, 2000, 0, 43, 46, 0, 0, 2347]
Robot Data: [56, 80, 139, 53, 80, 96, 144, 124, 20, 16, 2000, 0, 50, 46, 0, 0, 2348]
Robot Data: [60, 84, 138, 57, 81, 95, 143, 125, 20, 16, 2000, 0, 49, 46, 0, 0, 2349]
Robot Data: [64, 84, 135, 60, 82, 94, 141, 126, 20, 16, 2000, 0, 49, 46, 0, 0, 2350]
Robot Data: [67, 85, 136, 64, 83, 93, 140, 127, 20, 16, 2000, 0, 49, 46, 0, 0, 2351]
Robot Data: [70, 85, 133, 68, 83, 93, 139, 127, 20, 16, 2000, 0, 49, 46, 0, 0, 2352]
Robot Data: [74, 87, 137, 71, 84, 93, 139, 128, 19, 16, 2000, 0, 51, 52, 0, 0, 2353]
Robot Data: [78, 85, 137, 76, 84, 93, 139, 129, 20, 16, 2000, 0, 48, 48, 0, 0, 2354]
Robot Data: [84, 85, 140, 81, 84, 93, 139, 130, 20, 16, 2000, 0, 46, 50, 0, 0, 2355]
Robot Data: [90, 85, 146, 86, 84, 93, 140, 131, 20, 16, 2000, 0, 49, 48, 0, 0, 2356]
Robot Data: [96, 91, 160, 92, 85, 93, 144, 133, 20, 16, 2000, 0, 46, 50, 0, 0, 2357]
Robot Data: [104, 91, 172, 100, 86, 93, 150, 135, 19, 16, 2000, 0, 53, 52, 0, 0, 2358]
Robot Data: [109, 95, 177, 107, 88, 93, 155, 138, 20, 16, 2000, 0, 47, 51, 0, 0, 2359]
Robot Data: [117, 107, 185, 116, 92, 94, 161, 141, 20, 16, 2000, 0, 49, 49, 0, 0, 2360]
Robot Data: [125, 113, 187, 126, 96, 95, 166, 144, 20, 16, 2000, 0, 49, 49, 0, 0, 2361]
Robot Data: [135, 126, 186, 136, 102, 97, 170, 147, 21, 16, 2000, 0, 45, 45, 0, 0, 2362]
Robot Data: [146, 126, 177, 145, 107, 99, 171, 149, 20, 16, 2000, 0, 50, 46, 0, 0, 2363]
Robot Data: [154, 130, 178, 159, 112, 101, 172, 151, 20, 16, 2000, 0, 50, 47, 0, 0, 2364]
Robot Data: [167, 123, 189, 177, 114, 102, 175, 153, 20, 16, 2000, 0, 48, 49, 0, 0, 2365]
Robot Data: [183, 123, 207, 195, 116, 103, 181, 156, 19, 16, 1720, 0, 35, 39, 0, 0, 2366]
Robot Data: [198, 132, 230, 216, 119, 105, 191, 161, 19, 16, 1440, 0, 21, 20, 0, 0, 2367]
Robot Data: [214, 147, 258, 244, 125, 108, 204, 167, 16, 16, 1160, 0, 14, 14, 0, 0, 2368]
Robot Data: [228, 163, 279, 270, 133, 111, 219, 174, 14, 16, 880, 0, 6, 2, 0, 0, 2369]
Robot Data: [241, 180, 293, 296, 142, 115, 234, 181, 13, 16, 600, 0, 5, 9, 0, 0, 2370]
Robot Data: [251, 188, 294, 316, 151, 120, 246, 188, 10, 16, 320, 0, 13, 16, 0, 0, 2371]
Robot Data: [266, 198, 269, 338, 168, 130, 255, 198, 11, 16, 0, 0, 16, 12, 0, 0, 2373]

"""

In [4]:
sensor_df = dataframe_from_pasted_log(pasted_log)
display(sensor_df.head())
print(f"Parsed {len(sensor_df)} rows")

output_csv = next_dated_csv_path(data_dir)
sensor_df.to_csv(output_csv, index=False)
print(f"Saved to: {output_csv}")

,Front Left,Left Angle,Right Angle,Front Right,edgeLeftFast,edgeLeftSlow,edgeRightFast,edgeRightSlow,Distance Count,Maze Location,...,Distance Count Travelled,Distance MM,Profile Forward Speed mm,Profile Rotation Speed deg,LeftVolts,RightVolts,Maze X,Maze Y,time_s,Speed_mm_s
0,37,96,10,31,118,148,11,27,20,2,...,20,3.556732,355.673168,0.0,1.620,1.908,0,2,0.00,0.000000
1,38,98,10,35,114,145,11,26,20,3,...,40,7.113463,355.673168,0.0,1.800,1.728,0,3,0.01,355.673168
2,38,101,12,35,111,142,11,25,20,3,...,60,10.670195,355.673168,0.0,1.908,1.620,0,3,0.02,355.673168
3,42,104,13,38,110,140,11,24,20,3,...,80,14.226927,355.673168,0.0,1.872,1.656,0,3,0.03,355.673168
4,42,109,14,40,110,138,12,23,20,3,...,100,17.783658,355.673168,0.0,1.944,1.584,0,3,0.04,355.673168


Parsed 767 rows
Saved to: data/20260312_2.csv
